# Домашняя 3 · RAG: чанкование, бюджет контекста и метрики без судьи

**Неделя 12 · занятие 1.** Опора — L15 «Основы RAG · Чанкование · Понимание запроса»
и L16 «Оценка RAG · Агентный RAG».

RAG — это поиск плюс генерация, и почти всё, что в нём ломается, ломается в поиске.
Сегодня мы режем документы на чанки и обнаруживаем, что вопрос «стало лучше или хуже»
не имеет одного ответа: он **зависит от того, что именно ты считаешь найденным**.

| # | вопрос занятия | чем отвечаем |
|---|---|---|
| 1 | Что чанкование делает с полнотой? | меряем две разные полноты и получаем противоположные ответы |
| 2 | Как выбрать размер чанка? | сравниваем при **фиксированном бюджете контекста**, а не при фиксированном k |
| 3 | Как оценивать RAG, не имея судьи? | считаем четыре метрики RAGAS вручную и сверяем с лекцией |

**Данные.** 20 Newsgroups, две тысячи документов длиннее 150 слов, псевдозапросы того же
устройства, что на неделях 7–10. Игрушки из `data/l10-*.json` и `data/l11-*.json` для сверки.

**Среда.** Colab T4 через VS Code. Языковая модель сегодня **не нужна**: весь семинар —
про поисковую половину RAG и про метрики, которые считаются без судьи. Генерация упоминается
цитируемыми числами и разбирается в нижнем слое.

**Бюджет: ≈120 минут.**

**Артефакт на вынос.** `artifacts/rag.json` — выбранная конфигурация чанкования с её полнотой
и бюджетом. На неделе 14 это первая ступень твоего проекта.

**Как запускать.** Сверху вниз. Кодирование чанков — самая тяжёлая часть, около двух минут
на CPU.

<details><summary>Почему в RAG почти всё ломается в поиске, а не в генерации</summary>

Соблазн очевиден: если ответ плохой, виновата модель. Обычно виноват контекст, и вот три
причины, по которым это так.

**Первая: модель не может ответить тем, чего не видела.** Если нужный фрагмент не попал
в контекст, лучшая на свете языковая модель либо честно скажет «не знаю», либо придумает.
Оба исхода — следствие поиска, а не генерации. Это тот же потолок, что на неделях 7, 9 и 10:
**последняя ступень ограничена первой**, и сегодня мы встречаем его в четвёртый раз.

**Вторая: контекст конечен.** В окно помещается несколько килобайт, и это принудительный
отбор. Даже если поиск нашёл двадцать релевантных чанков, в промпт влезет пять, и выбор пяти
из двадцати — снова задача ранжирования.

**Третья: лишний контекст вредит.** Модель, получившая пять релевантных чанков и пятнадцать
случайных, отвечает хуже, чем получившая пять релевантных. Шум в контексте отвлекает,
и эффект измерим. То есть точность отбора важна не меньше полноты — в отличие от каскада,
где первая ступень отвечала только за полноту.

**Что из этого следует для сегодняшнего занятия.** Мы будем мерить поиск, а не генерацию,
и это не упрощение ради времени. Это правильный порядок: пока не измерена доля запросов,
где нужный фрагмент вообще попал в контекст, обсуждать качество ответов бессмысленно.
</details>

## Шаг 0 · Пины и preflight

In [ ]:
# ПИНЫ — точнее, ОТКАЗ от них там, где они ломают Colab.
# Базовый стек образа (numpy, scipy, scikit-learn, matplotlib, torch) собран сам под себя.
# Понижать его нельзя: `pip install numpy==1.26.4` откатывает ОДИН numpy, а scipy и sklearn
# остаются собранными под numpy 2 — и первый же импорт падает с
# «ModuleNotFoundError: No module named 'numpy.char'». Ставим ТОЛЬКО то, чего в образе нет.
import importlib.util as _ilu, subprocess as _sp, sys as _sys

NEEDED = {"sentence_transformers": "sentence-transformers"}
_missing = [pkg for mod, pkg in NEEDED.items() if _ilu.find_spec(mod) is None]
if _missing:
    print("ставлю:", ", ".join(_missing))
    _sp.run([_sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    print("готово · если следующий импорт упадёт — Runtime → Restart session, потом эта ячейка снова")
else:
    print("всё нужное уже в образе Colab — ставить нечего")

import json, math, os, random, re, statistics, time
from pathlib import Path
from IPython.display import HTML, display   # ядро Jupyter/Colab: есть всегда,
                                            # ставить нечего, версия — версия ядра.
# google.colab (drive.mount) — часть рантайма Colab, не пакет: вне Colab его просто нет,
# и ноутбук честно ловит ImportError и пишет артефакты рядом с собой.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

In [ ]:
# ДАННЫЕ ЛЕКЦИИ. Занятие сверяет свои результаты с числами лекции, а те живут в папке
# data/ курса. В Colab её нет — и раньше ноутбук падал на первой же сверке с
# FileNotFoundError. Файлы крошечные, поэтому вшиты прямо сюда: ниже точная копия нужных
# data/*.json, сжатая zlib и записанная base64. Гейт _research/check_notebooks.py следит,
# чтобы копия совпадала с оригиналом байт в байт, — так что разойтись они не могут.
# Если рядом уже лежит настоящая папка курса (или задан DLS_DATA), берётся ОНА.
import base64 as _b64, os as _os, zlib as _zlib
from pathlib import Path as _Path

_DATA_DIR = _Path(_os.environ.get("DLS_DATA", "./data"))
_EMBEDDED = {
    "l10-budget.json": (
        "eNqNUsFuGjEUvOcrRlxC1AWWJRBA6iGqKqWHtFKL1ENVIbP7AAuzTmwv7CpK1VPVc9Qv5Ev67N1KlEZRLzaeZebNe28e"
        "zoDWPNNpa4rWTG8o7yyKbEUOdk90F4FKkTq0rcuUXFx0MVsTPl3fvsVSlpRB78isSWQQFo4/iTxdawNnREo4fP8FW1lH"
        "WyRxjFe4L8hUGPifw34CQ5bMjlWWnhLYXNVESIajjvNmkK6LfGOD0qJwEKnR1jJRqM5eG5Uh1bmj0mEv80zvLS7jyQg9"
        "jPuThK9BcjUa891PxnEcd7G5FSVeo526EoefT95duO/D2fi5QK9XF8bhxxP6AxZIJkFlyOfl5KqLayzkakWmqcvmWGmr"
        "DTWOo2A3lFsZ7+ubkjkJoypI7sqVEd5/mIXGl4ZoCtr50XAnRjS1ZSOohKM8rXhm4XmutHUdmXd4YJ2tzDJF58ikKpzU"
        "eQSrYUS+kfkK1kmlkFEqM7L4fPPuzQ02qHSBDa+224rC7q0uTEp+/XPfvzDpureifK76cfeuQtvpilXr9U+fGyCvNwSn"
        "maRf8dG7XunH08nW/2g8HIuwEc5KgI+kGB3U4F96DHOOAn6k6iWGo4DyfLeFEr67/915bamJExO/8BN4CKev40oGfcqi"
        "P5CfCWP9QQAeo+cIPo+nhGTyAiEk958SyfAFSh3yUw7Htebw+fXs8ew3eAY0KQ=="
    ),
    "l10-rag.json": (
        "eNqlVs1u20YQvvspBrpUQiXq37IVBIXRFs2hSRDXQFEEhrEi1yIrksvsLiWrgYGcip6Doq+Sex7FT9Jvdkn92HLrpgdR"
        "3NmdmW/+Pu77I6LGVaTCxpQa52c/kMjDWGmyWoSSviarFjKnWRnNpQ3oQq2pKW9EaNtkbJQms9aUQpVbeWNp1Ds9fkZm"
        "bazMaNDrQftdKfWahvy6+PxpMD7ueHthXOYL0yYtjdTLJJ/TuD+ga/YbS0AwK6kDOrO0eD6iu98/wll5fS0j6vcGozYV"
        "WmWFpf5gjEUsRaSVgsuT0SCgxUtxQ8/pOlVKN5uM6e6Pj0CD55AfcNTqAklrc2g4HI9qSX9Idx/+dChWSR6pFYWiMBTj"
        "JSvDmNaqhCQHcKsTuZTUXMVr0iJfcBCZsFZq0wro9avv+ZzKk1CkJPMIgXfwV+U1Mc5FKkNbavmVgb2w1JptVPm/TubY"
        "oSYf02LeKZJCpknOsLgUXsFgZWNSEBsr5rCbI4lhaSCiMc2ksAwmTJNs1nE59x6iSuTKo+VKJxZe5LXUMgc6GwOfERlk"
        "ZZ6zCkqeFakM6FwiHCROq9/kNguo0FzmUguYcQGiO4o0CVGw2ZquuMpCh3EXh67Sfg8CkQbFOmi0XfcZVepQcgM+OIpT"
        "yIFab9oNnXTPCjX55RCWNnVLo7uzJO8WaxurfNjyHquO/dlVGH65S9yGb94L7lEDOdrGiV2eNtKhF/ouPXcdzODRWN44"
        "53lrYlxZ5vqw4C1W9SmHBguZzWRUL+pA6rVr/XpRh9bA8tIZXim9gPKU3vsTC45nV9VturnxQj88F8qKlDd4hvxGPUiM"
        "GqME4a3zwBPFJ4du5eq7dedSw6V7gRGJlPSNDVPaUlFmBc0wY9E394OLNpmgypLbTljeABtdhUJHicD/OkzrVPiYQqXZ"
        "fy84GeyIfT6hfAH3lTI5ZRIphjJH0owjJ8UijABOuMUzWsocqMIyFdqRGSK0CSZI/or5ND6CoFH5um3/I+xEsx2rkJSD"
        "oCenj4LeqtYkmqmlrPwjrVqV85iQWIkUGhfDUiY5Mq5c0mcqWoNKbGJMKc0TAS9FCh+PYO0fwvrC1dbr0ULKqsRMpivm"
        "CmajlViDfexKgiQcoyO/okLscp0+GaBA5VT2SDaPx49l03dgLAwIsURZY4EZ02ZKdqV24PDqECT3f7k3MWz5J1eXKZ25"
        "2afScMCvX/34i4uyopWAvvUvaPFR9am7DOgNisJtNaWnjYrnl/14+Nj9hrD3G35KUek+JXW384ltkJsWdwlwPe4ttmkV"
        "Jzi+W9rEPqjr2wfj2aZtJ126HHriqIZdpC+FjbeUESWZ2Rn/BgfV3Zm7TakbVe27xurSfSm3W4yqC39GptvubXgMXUDt"
        "AGpjr4o1U9V+h7XWhkY27d7fU8yVzt5UysPgZFIpcneaL6axpQx3dPe872F6uLN5v9wxFynrWH5HxLC/c/e6YdA/Hgx3"
        "J0WZkafQ/v4EKVMx6/8hu/8S2uDJofUfCe2457+w90ObnEwmB0KbnH4hLf5LVOPdRe/JUY0PRjUORpPJ4GBUvX1SrqPq"
        "HyQuXKgzFIiDwknc2+C1+a4dtahLzbsPf73D7/MnPCL8Wo1Nv1t3GRt0ooJYT0swYFSGzFj89Q82X3FyVGxcy3Q5ufzo"
        "uys0LqhgzdLiEG6LGS6xK5BQiGbBvT+HsgY7415cJCHuL541jm6P/gY8YI7e"
    ),
    "l11-judge.json": (
        "eNp9VNtqGzEQffdXDPsSm7rG9xY/FDYmveHS4ISGUkKQd8e7KlrJlbTZuCHQp35AKfSD+if5ko609q6dODWLGY00Z0Zn"
        "jua2ARBcGR0FEwisWoPOF5pHwDS3aYaWzGaC8kr0ep3VugXPYH4SzmCp1XeUIATL2KDTm7xcwNc8TpDCpQnaDrQEIthb"
        "WtE6IkTUnJHni/e4MyjwmskIfYh3JVrlMuYyqV2RylYCLUo0JvDOy3IvMBET+IFLwuztudgNuUbkufO1MGkK1KZKfbuF"
        "5rG7d1jnMpHSaHZqBBi1K3NYm4ONdVmFZshcIcNO13vu2odSHf831Q5+f8d8KlW/Mx6PX5TZGpv9YMW4LrjBCy4l6vp6"
        "QaJUnDJt65akiiit15sz/g4D+tU9YRmW7m7lK3bgd+8bCJSJTY85Mz7mMXT3AG5/dAB4ug/sAM4eclbFVcZgTyE+xaOo"
        "iufqjqO9KKksugreeoLA0T2BEJolLy14BVO36LY68PEa9fMCeZJaONpV6hH8/dOHJoOVVjdrWCoNJTUtYDIuAfqjFiyQ"
        "WVOCd1sTsClSPuve4AIjqh4YWKYTtB04T7kB+t5sOnlkQLAC7n/8hlnv/uev2dD90UsNKuVrZKJu+FWmYnTrYOflbhQZ"
        "sCjKNYvWU4HM8d/bdiqQW9d441gpwy1X8rUSQhVzZvGcO8K6XpAPDoUySon/uDwy3GKWyy0isbhw59enGpeokWaCg90r"
        "ozozUzJBTQJ3Pe1VVX7a7r93oyhDaZ/YP6Un4rY2mgmWjIbdMheubVOWUytPBZPWa9PqHB+oYk6cHhh9TYvZCrptMIgx"
        "DPskjtBTahGKlKblt5wJyu46OJ2dhHNo0t1abVjkliThSDJgFRihLNAMRE2tP393As2S1lI34ewi/HxGonI0GS8X4emA"
        "csiVoF4TW5m0nYBNTi0ohdG4a/wDWvqTLA=="
    ),
    "l11-ragas.json": (
        "eNq1Vc1u00AQvvcpRjmBFEISlzRwQSVItOICpYIDQtVmPY6XrnfN7jqtVVXiIXhCnoTZdeI4iQ1tJC7+mZ2fz983M747"
        "AuhdWcN7r6DndAk32lxjDHjLslwiPFmgupKj0SAv+2BdLMX8Kfz++QsuTt+dfoJEFwYydEZw24eUqfgZT5FfsznFJoZx"
        "J7SyzzNkyvb6vtaPAq03+npn+gZijRZcipAiMw7yIsthLrWOX1f+XCuHt86S/1d6B7gLVzoRsc/BR8EvWAxT12QbbQwo"
        "ccmUI6MzBdZ2n9EHX9Z1hQUGWWF5IZkBbRZMESpWAbIVIjIYXSzSAJcL432dNiXY0jrMBr2Q/r7fCnO8B3PcBjNh0u7j"
        "fFOVL3P0QGN0aDKhSKZ5CRQnSCQLWgVgtjAJ4wg6AeM9QihHKe3fAUZ7AKMH8/gZlW+BwJ0nQ5P6XjjfAJW6y8pDIrUJ"
        "fkfqC7VYYRPK6eBDOqAR+A+cx3s4jx9OpBecLdGwBd3jQjpIi4ykTpmFoK0sIRFLBCmcocYkEgPINSS6fgttSf18g2Ym"
        "mchaWvOw/hpsvswWea6JjnjFeCsh6yrnrkG2SUuXZoIzKctDE7bJGUSz3ZIdVumLkFKwDM6YWWIJsbBckzzoZ40oWw8Z"
        "7QuqCaPJeNpRKci9o9GCFFXxpSlc+nChuuZ9ruMmn0LNqsX03/l8VKW3glXpKZmi+U+I3+Z65SnL5mi68rexaJAEsfhx"
        "tbdnekPjcPBytcSGg+l0/RSN6th6OunX4pi82IxniOvlBrmwlPXUvd8XZ3uX7/iOBsOu9dTK0PbG3Uk2HLzo3CGd2aLu"
        "bJPJ5OTR6I4PR1cTnjDh0qSQCq0NkScvGutqxT/H6mgYRc1f7Id11XA6jbZPL9CvlK2v2wxgPV7R7nKc0Qi6ugsaE3ne"
        "aLvx3riuoqKj+6M/u8h0iA=="
    ),
}

_DATA_DIR.mkdir(parents=True, exist_ok=True)
_new = [n for n, b in _EMBEDDED.items() if not (_DATA_DIR / n).exists()]
for _n in _new:
    (_DATA_DIR / _n).write_bytes(_zlib.decompress(_b64.b64decode(_EMBEDDED[_n])))
print(f"данные лекции: {len(_EMBEDDED)} файл(ов) в {_DATA_DIR} · "
      f"распаковано {len(_new)}, остальные уже лежали на месте")


In [ ]:
def preflight():
    problems = []
    for name in ("l10-chunking", "l10-rag", "l10-budget", "l11-ragas", "l11-judge"):
        if not Path(f"{DATA_DIR}/{name}.json").exists():
            problems.append(f"нет {DATA_DIR}/{name}.json -- сверка с лекцией невозможна.")
    if SMOKE:
        print("· SMOKE: корпус и число конфигураций урезаны, числа НЕ сравнимы с разбором")
    for p in problems:
        print("!", p)
    print("preflight:", "ЧИСТО" if not problems else f"{len(problems)} замечани(я/й) -- читай выше")
    return not problems

## Шаг 1 · Конфигурация

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SMOKE = os.environ.get("SMOKE", "0") == "1"
N_DOCS = 400 if SMOKE else 2000
N_QUERIES = 40 if SMOKE else 150
CONFIGS = [(64, 0), (64, 16), (128, 0), (128, 32),
           (256, 0), (256, 64), (512, 0), (512, 128)]      # (размер, перекрытие) в словах
BUDGET_WORDS = 1024                                        # бюджет контекста для части 2
DATA_DIR = os.environ.get("DLS_DATA", "./data")
DRIVE_DIR = "/content/drive/MyDrive/dls-2026"   # накопительная папка курса (правило 10.4)


def _artifacts_dir():
    """Куда класть индекс, эмбеддинги и прочее, что подхватят СЛЕДУЮЩИЕ занятия.

    На Drive, а не в песочницу: песочница Colab умирает вместе с сессией, и цепочка
    занятий рвётся — четвёртое занятие уже не найдёт индекс третьего и молча соберёт
    уменьшенный свой. На Drive артефакты переживают и перезапуск рантайма, и неделю
    между парами, так что к концу курса собирается одна система, а не семь огрызков.
    """
    if os.environ.get("ARTIFACTS"):            # явное указание сильнее всего
        return Path(os.environ["ARTIFACTS"])
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive", force_remount=False)
        return Path(DRIVE_DIR) / "artifacts"
    except Exception as _e:                    # не Colab либо отказ в доступе — не беда
        print(f"Drive не подключён ({type(_e).__name__}): артефакты лягут рядом с ноутбуком.")
        print("Занятие отработает целиком, но следующее не подхватит их и соберёт своё.")
        return Path("./artifacts")


ARTIFACTS = _artifacts_dir()
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Журнал прогона. Всё, что печатают ячейки ниже, дублируется в файл и в конце занятия
# уезжает архивом по ссылке: тетрадка, сохранённая без выходов, теряет диагностику.
# Две тонкости, обе проверены на живом прогоне:
#   · подменять sys.stdout целиком НЕЛЬЗЯ — ipykernel опознаёт поток по типу, и подмена
#     уносит весь вывод мимо тетрадки: ячейки остаются пустыми, студенту смотреть не на что;
#   · патч .write надо ставить ЗАНОВО перед каждой ячейкой — IPython сам оборачивает write
#     на время ячейки (складывает вывод в историю) и снимает обёртку после, поэтому
#     одноразовый патч молча перестаёт писать со второй ячейки.
import sys as _sys

NB = "hw-rag"
RUN_DIR = ARTIFACTS / "runs" / NB
RUN_DIR.mkdir(parents=True, exist_ok=True)
_LOG = open(RUN_DIR / "run_log.txt", "w", encoding="utf-8")


def _arm_log(*_a):
    _st = _sys.stdout
    if getattr(_st.write, "_dls_log", False):
        return
    _orig = _st.write

    def _w(data, *a, **k):
        try:
            _LOG.write(data)
        except Exception:
            pass
        return _orig(data, *a, **k)

    _w._dls_log = True
    _st.write = _w


try:
    from IPython import get_ipython as _gi
    _ip = _gi()
    if _ip is not None and not getattr(_ip, "_dls_log_armed", False):
        _ip.events.register("pre_run_cell", _arm_log)
        _ip._dls_log_armed = True
except Exception:                      # не IPython (прогон файлом) — журнал всё равно пишется
    pass
_arm_log()
print(f"журнал прогона: {RUN_DIR / 'run_log.txt'}")

RUN = {"seed": SEED, "smoke": SMOKE, "n_docs": N_DOCS, "n_queries": N_QUERIES,
       "budget_words": BUDGET_WORDS}
print(json.dumps(RUN, ensure_ascii=False))
preflight()

**Что видно.** Конфигурации заданы парами «размер, перекрытие» в **словах**, а не в токенах,
и это первое решение, о котором надо знать. Сравнивать надо не пары между собой, а **единицу
измерения с тем, чем меряет модель**: языковая модель считает токены, а мы режем по словам,
и одно к другому не сводится постоянным множителем. Механизм: токенизатор дробит редкие слова
на куски, и текст с терминами даёт больше токенов на слово, чем обычная проза. Чего этот вывод
НЕ показывает: величины расхождения — мы её сейчас измерим. Что делать: держать в голове, что
все «1024 слова» ниже — это примерно, и уточнить это «примерно» первым делом.

---

## Часть 1 · Что чанкование делает с полнотой — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 1.1 | Как устроен наш эксперимент? | строим корпус, запросы и разметку по позиции |
| 1.2 | Сколько токенов в слове? | считаем расхождение токенизатора и пробелов |
| 1.3 | Что происходит с полнотой? | меряем две разные полноты и получаем разные ответы |

<details><summary>Почему бюджет контекста считают не «сколько влезет», а с тремя резервами</summary>

Формула из лекции выглядит бухгалтерски: окно минус система, минус запрос, минус резерв
под ответ. Каждое вычитаемое существует по своей причине, и пропуск любого приводит
к неприятностям в проде.

**Системный промпт.** Инструкции о том, как отвечать, чего не делать, каким тоном. Обычно
двести-пятьсот токенов, и он **растёт** со временем: каждый инцидент добавляет строку.
Планировать бюджет по сегодняшнему системному промпту — значит через полгода получить
переполнение.

**Запрос.** Кажется мелочью, пока запросы короткие. В агентных сценариях в промпт попадает
история диалога, и «запрос» становится тысячами токенов.

**Резерв под ответ.** Самый забываемый. Модель генерирует в то же окно: если под ответ
не оставлено места, он оборвётся на середине. Пятьсот токенов — разумный минимум, для развёрнутых
ответов нужно вдвое больше.

**Что происходит при переполнении.** Библиотеки обрезают — и обрезают обычно **начало или конец**
промпта, то есть либо системную инструкцию, либо самый релевантный чанк, если он последний.
Ошибки не будет, ответ просто станет хуже. Это ловушка типа D в чистом виде: код работает,
качество тихо падает.

**Практика.** Считать бюджет заранее, оставлять запас процентов в двадцать, и **логировать
фактическую длину промпта** на каждом запросе. Гистограмма этой длины за неделю скажет о системе
больше, чем любая метрика качества: если она упирается в потолок, у тебя уже режется контекст,
и ты об этом не знаешь.
</details>

### Шаг 1.1 · Корпус, запросы и золотой чанк

Конструкция та же, что на неделях 7–10: предложение вынимается из документа и становится
запросом. Новое — то, что документ теперь режется, и **золотым считается чанк, покрывающий
позицию, откуда предложение вынули**.

In [ ]:
raw = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"),
                         random_state=SEED)
TOKEN = re.compile(r"[a-z]{2,}")
SENT = re.compile(r"(?<=[.!?])\s+")
pool = []
for t in raw.data:
    t = " ".join(t.split())
    if len(TOKEN.findall(t.lower())) < 150:      # длиннее, чем раньше: чтобы было что резать
        continue
    ss = [s for s in SENT.split(t) if 12 <= len(s.split()) <= 25]
    if ss:
        pool.append((t, ss))
random.Random(SEED).shuffle(pool)

N = min(N_DOCS, len(pool))
DOCS, QUERIES, GOLD_W = [], [], []
for i, (text, ss) in enumerate(pool[:N]):
    if i < N_QUERIES:
        s = max(ss, key=len)
        pos = text.index(s)
        doc = (text[:pos] + text[pos + len(s):]).strip()
        DOCS.append(doc)
        QUERIES.append(s.strip())
        GOLD_W.append(min(len(text[:pos].split()), max(0, len(doc.split()) - 1)))
    else:
        DOCS.append(text)

lens = np.array([len(d.split()) for d in DOCS])
print(f"корпус: {len(DOCS)} документов · запросов: {len(QUERIES)}")
print(f"длина документа: медиана {np.median(lens):.0f} слов, "
      f"25-й и 75-й перцентили {np.percentile(lens, 25):.0f}--{np.percentile(lens, 75):.0f}")
print(f"позиция вынутого предложения: медиана {np.median(GOLD_W):.0f}-е слово "
      f"(то есть примерно {np.median(GOLD_W) / np.median(lens):.0%} документа)")
RUN["n"], RUN["median_doc_words"] = len(DOCS), float(np.median(lens))

**Что видно.** Документы существенно длиннее, чем на прошлых занятиях: медиана в несколько сотен
слов, и разброс велик. Сравнивать надо не медиану с чем-либо, а **межквартильный размах
с размером чанка**: при чанке в 256 слов короткие документы дадут один чанк, длинные — пять,
и это уже само по себе смещает поиск в пользу длинных, потому что у них больше «билетов»
в выдаче. Механизм прямой и его надо помнить: чанкование меняет не только гранулярность,
но и **число представителей** каждого документа. Чего эта статистика НЕ показывает: где внутри
документа лежит ответ — третья строка отвечает и на это: примерно в середине, то есть
не на границе, что для нас удобно. Что делать: заметить, что документы разной длины после
чанкования становятся неравноправными, и вернуться к этому в разборе полноты.

<details><summary>Что мы построили за курс — и как это складывается в одну систему</summary>

Неделя 12 — последняя техническая, и стоит собрать конвейер целиком, потому что на неделе 14
его придётся защищать.

**Неделя 3, лексический поиск.** Инвертированный индекс и BM25 руками. Артефакт: индекс
и выдачи. Урок: лексический потолок — 64 % корпуса недостижимы запросом, сколько ни улучшай
формулу.

**Неделя 4, метрики.** Пять метрик, сверенных с лекцией, и статистика сравнения двух систем.
Артефакт: `metrics.py`, которым мерят все последующие занятия. Урок: интервал важнее `p`,
а среднее скрывает знаки.

**Неделя 7, каскад.** Би-энкодер и кросс-энкодер. Артефакт: эмбеддинги, выдачи и скоры.
Урок: первая ступень отвечает за полноту, вторая за точность, и потолок задаёт первая.

**Неделя 9, слияние.** RRF и взвешенное слияние скоров. Артефакт: конфигурация и разбиение
запросов. Урок: оптимизм подбора съедает половину эффекта, если не отложить данные.

**Неделя 10, приближённый поиск.** HNSW, IVF, PQ. Артефакт: рабочая точка индекса. Урок:
согласие с точным поиском и потеря конечной метрики — разные величины.

**Неделя 12, чанкование.** Артефакт: размер чанка и бюджет. Урок: две полноты с одним именем
дают противоположные ответы.

**Что получилось.** Конвейер: чанкование → индекс → лексический и плотный отбор → слияние →
переранжирование → контекст. Каждая ступень измерена, у каждой записан потолок, у каждой
названа цена. Это и есть содержание проекта недели 14: не «построить RAG», а **обосновать
числами каждое из шести решений**.
</details>

<details><summary>Понимание запроса: четыре приёма, которые дают больше, чем размер чанка</summary>

Всё занятие мы крутим одну ручку — размер чанка. На стороне **запроса** ручек больше, и они
часто дают больше, потому что запрос короче документа и его легче исправить.

**Переписывание (query rewriting).** Короткий бедный запрос переписывается моделью в развёрнутый.
`data/l10-rewrite.json` показывает случай, где исходный запрос находит нужный документ
на восьмом месте, а переписанный — в первой пятёрке. Стоит одного вызова модели на запрос,
то есть заметно дешевле, чем переиндексация корпуса.

**HyDE.** Гипотетический документ: модель пишет **ответ** на запрос, и ищется он, а не запрос.
Идея в том, что ответ лексически и семантически ближе к документу, чем вопрос. Работает
особенно хорошо там, где вопрос и документ написаны на разных «языках»: пользовательский
вопрос против технической документации.

**Декомпозиция.** Составной вопрос («чем отличается A от B и что дешевле») разбивается
на подвопросы, каждый ищется отдельно, результаты объединяются. `data/l10-decomp.json`
показывает, почему это нужно: совместная полнота по составному вопросу ниже, чем полнота
по каждой части отдельно, — один вектор не может быть близок к двум разным темам сразу.

**Мультизапрос плюс слияние.** Генерируем несколько перефразировок, ищем каждой, сливаем
через RRF — тот самый RRF с недели 9. Дёшево, устойчиво, почти всегда даёт прирост полноты.

**Почему это часто выгоднее, чем возиться с чанками.** Изменение чанкования требует полной
переиндексации корпуса — часы или дни. Изменение обработки запроса не требует ничего:
выкатил и откатил. Правило приоритетов: **сначала крути то, что не требует переиндексации.**
</details>

<details><summary>Четыре стратегии чанкования — и почему фиксированное окно всё ещё побеждает</summary>

Мы режем по фиксированному числу слов. Это самая примитивная из четырёх стратегий, и она же
самая распространённая. Разберём, почему.

**Фиксированное окно.** Режем каждые `N` слов, игнорируя структуру. Просто, предсказуемо
по размеру, работает везде. Ломает предложения и абзацы пополам — главный и единственный
недостаток.

**По границам предложений.** Набираем предложения, пока не упрёмся в лимит. Чанки перестают
рваться посреди мысли, размеры становятся неравномерными. Требует надёжного разбиения
на предложения, а оно ломается на аббревиатурах, списках и коде.

**По структуре документа.** Заголовки, абзацы, разделы. Лучший вариант, когда структура есть
и размечена: markdown, HTML, документация. Бесполезен на плоском тексте вроде наших почтовых
сообщений.

**Семантическое.** Кодируем предложения, ищем места, где соседние предложения расходятся
по смыслу, режем там. Звучит идеально, стоит одного прохода энкодера по всему корпусу
до индексации, и выигрыш на практике оказывается небольшим — обычно единицы процентов
относительно фиксированного окна.

**Почему фиксированное окно держится.** Три причины. Первая: предсказуемый размер, а значит
предсказуемый бюджет контекста — попробуй спланировать, сколько чанков влезет, если их размеры
гуляют втрое. Вторая: отсутствие зависимостей — не нужен ни парсер, ни модель. Третья и главная:
разница в качестве между стратегиями обычно меньше, чем разница между размерами внутри одной
стратегии, а размер подобрать проще.

**Что действительно помогает.** Не выбор стратегии, а **перекрытие** и **обогащение чанка
контекстом**: приписать к каждому чанку заголовок документа и, скажем, первое предложение.
Дёшево, работает почти всегда и почему-то делается редко.
</details>

### Шаг 1.2 · Замер без модели: слова против токенов

**Замер без модели.** Прежде чем говорить о бюджете контекста, надо понять, в чём он
измеряется. Модель считает токены, мы режем по словам, и множитель между ними — не константа.

In [ ]:
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
tok = st.tokenizer

sample = DOCS[:200]
ratios = []
for d in sample:
    w = len(d.split())
    t = len(tok.encode(d, add_special_tokens=False))
    if w:
        ratios.append(t / w)
ratios = np.array(ratios)

print(f"токенов на слово: медиана {np.median(ratios):.2f} · "
      f"5-й и 95-й перцентили {np.percentile(ratios, 5):.2f}--{np.percentile(ratios, 95):.2f}")
print(f"то есть 1024 слова -- это примерно {1024 * np.median(ratios):.0f} токенов, "
      f"а в худшем случае {1024 * np.percentile(ratios, 95):.0f}")
print()
CW = json.load(open(f"{DATA_DIR}/l10-rag.json", encoding="utf-8"))
overhead = CW["systemTokens"] + CW["queryTokens"] + CW["answerReserve"]
print(f"окно контекста из лекции: {CW['contextWindow']} токенов")
print(f"накладные (система {CW['systemTokens']} + запрос {CW['queryTokens']} + "
      f"резерв под ответ {CW['answerReserve']}) = {overhead}")
print(f"под чанки остаётся: {CW['contextWindow'] - overhead} токенов")
RUN["tokens_per_word"] = float(np.median(ratios))
# Разброс важнее медианы: на него ссылается вывод «сравнивать надо 95-й перцентиль
# с медианой», поэтому оба перцентиля уезжают в дамп, а не только печатаются.
RUN["tokens_per_word_pct"] = {"5": round(float(np.percentile(ratios, 5)), 4),
                              "95": round(float(np.percentile(ratios, 95)), 4)}
# Сами уровни — тоже числа разбора: проза говорит «сравнивать 95-й перцентиль с медианой».
RUN["pct_levels"] = [5, 50, 95]
# Сколько СЛОВ помещается в окно энкодера — число, на котором стоит вывод про обрезку
# документа («энкодер видит примерно первые N слов»). Считаем, а не прикидываем в тексте.
RUN["encoder_window_tokens"] = int(st.max_seq_length)
RUN["encoder_window_words"] = round(st.max_seq_length / RUN["tokens_per_word"])
print(f"окно энкодера {RUN['encoder_window_tokens']} токенов -- это примерно "
      f"{RUN['encoder_window_words']} слов документа")

**Что видно.** Одно слово — это заметно больше одного токена, и разброс между документами
велик. Сравнивать надо не медиану с единицей, а **95-й перцентиль с медианой**: документ
с терминами, опечатками и адресами даёт токенов в полтора раза больше типичного, и если ты
планировал бюджет по среднему, такой документ его переполнит. Механизм: BPE дробит редкие
слова на куски, и чем специфичнее текст, тем сильнее. Чего этот замер НЕ показывает: что будет
с другим токенизатором — у каждой модели он свой, и множитель у GPT-подобных обычно ниже.
Что делать: считать бюджет **в токенах своего токенизатора**, а не в словах и не в символах.
Ошибка здесь приводит к обрезанному промпту, а обрезается обычно конец — то есть инструкция
о том, как отвечать.

<details><summary>Чанкование русского текста: что меняется</summary>

Весь наш корпус английский, и часть выводов на русском выглядела бы иначе. Курс идёт
по-русски, и разница стоит того, чтобы её назвать.

**Токенов на слово больше.** Мы намерили около 1,35 для английского. Для русского у типичных
многоязычных токенизаторов выходит 2–3, потому что кириллица дробится сильнее: слово
разбивается на морфемы или даже на пары символов. Практическое следствие: **тот же чанк
в словах занимает вдвое больше токенов**, и бюджет контекста в словах для русского текста
переоценивает вместимость вдвое.

**Границы предложений сложнее.** Сокращения с точками («т. д.», «г.», инициалы) ломают
наивное разбиение по `[.!?]` гораздо чаще, чем в английском. Если чанкуешь по предложениям,
нужен нормальный сплиттер, а не регулярка — наша регулярка на русском давала бы куски
из половины предложения.

**Морфология размывает лексическое совпадение.** Запрос «как настроить принтер» и текст
«настройка принтера» не пересекаются ни одним токеном при наивной токенизации. Для плотного
поиска это неважно, для лексической ступени гибрида — критично, и лечится лемматизацией
или стеммингом. Это тема лекции L19 целиком.

**Модели эмбеддингов.** Многоязычные модели на русском слабее одноязычных английских
на своём языке — это измеримо и стабильно. Для русского домена стоит смотреть на модели,
дообученные на русском (`rubert`-семейство и многоязычные `e5`), и обязательно сравнивать
их с BM25 на своих данных: разрыв часто оказывается меньше ожидаемого.

**Что не меняется.** Всё содержание сегодняшнего занятия: две полноты, бюджет контекста,
потолок, зависимость выбора от следующей ступени. Меняются числа, а не устройство вопросов.
</details>

<details><summary>Как оценить чанкование, не имея вообще никакой разметки</summary>

У нас была разметка по построению: мы сами вынули предложение и знаем, откуда. В реальном
проекте её нет, а решение о размере чанка принимать надо. Три способа, от дешёвого к дорогому.

**Первый: псевдозапросы, как у нас.** Возьми документы, вынь из каждого предложение, объяви
его запросом. Разметка получается бесплатно и в любом количестве. Смещение известно и названо:
лексическое пересечение завышено, стиль запроса не пользовательский. Но для **сравнения
конфигураций между собой** этого хватает, потому что смещение одинаково давит на все.

**Второй: сгенерированные запросы.** Просишь модель написать вопрос, на который отвечает
этот чанк. Ближе к пользовательским формулировкам, стоит одного вызова модели на чанк
(то есть дорого при большом корпусе — но выборки в тысячу чанков достаточно). Смещение другое:
модель пишет вопросы, на которые чанк отвечает **хорошо**, и потому трудные случаи в выборку
не попадают.

**Третий: логи.** Реальные запросы плюс клики или явная обратная связь. Единственный источник
без смещения формулировок, и он же самый медленный: логи надо накопить, а до запуска системы
их нет вовсе.

**Практический порядок.** До запуска — псевдозапросы, чтобы выбрать стартовую конфигурацию.
После запуска — логи, чтобы её перепроверить. Сгенерированные запросы — когда нужен быстрый
ответ по домену, где псевдозапросы явно не работают (например, короткие структурированные
записи, из которых нечего вынимать).

**Проверка, которую стоит сделать в любом случае.** Возьми двадцать своих запросов, посмотри
глазами на найденные чанки и спроси: **можно ли ответить, имея только этот текст?** Двадцать
минут работы. Половина проблем с чанкованием видна сразу — обрывки на середине предложения,
таблицы без заголовков, куски кода без контекста. Ни одна метрика этого не покажет.
</details>

⚠️ Ловушка D · **«Примерно четыре символа на токен» — эвристика для английского.** Для русского
множитель другой и обычно хуже: кириллица дробится сильнее. Планировать бюджет по этой
эвристике — значит систематически недооценивать длину, а расплачиваться обрезкой промпта
в самом неудобном месте.

### Шаг 1.3 · Число, с которым сравнивается всё остальное

`BASE` сегодня — поиск **без чанкования**: документ целиком, один вектор на документ. Всё, что
даст чанкование, будет сравниваться с ним.

In [ ]:
QEMB = st.encode(QUERIES, normalize_embeddings=True, show_progress_bar=False).astype("float32")
t0 = time.perf_counter()
DEMB = st.encode([d[:2000] for d in DOCS], batch_size=128, normalize_embeddings=True,
                 show_progress_bar=False).astype("float32")
DOC_ORDER = np.argsort(-(QEMB @ DEMB.T), axis=1)

def recall_doc(k):
    return float(np.mean([1.0 if i in DOC_ORDER[i, :k] else 0.0 for i in range(N_QUERIES)]))

BASE = {k: recall_doc(k) for k in (1, 3, 5, 10)}
print(f"кодирование документов: {time.perf_counter() - t0:.0f} c")
print(f"BASE -- без чанкования: " + " · ".join(f"R@{k} {v:.3f}" for k, v in BASE.items()))
print(f"\nвнимание: документ обрезан по 2000 символов -- у длинных документов хвост "
      f"в эмбеддинг НЕ попал")
RUN["base"] = BASE

**Что видно.** Без чанкования нужный документ попадает в пятёрку примерно в шести случаях
из десяти. Сравнивать надо не эти числа с чем-либо внешним, а **последнюю строку с самой
идеей базовой линии**: документ обрезан по две тысячи символов, потому что у энкодера окно
в 256 токенов, и всё, что дальше, он не видит **вообще**. То есть наша «база без чанкования»
на самом деле уже и есть чанкование — с размером чанка «сколько влезло» и выбрасыванием
остального. Механизм тут принципиальный: **энкодер физически не умеет кодировать длинный
документ**, и чанкование — не оптимизация, а необходимость. Чего этот замер НЕ показывает:
сколько мы потеряли на обрезке — это стоило бы измерить отдельно. Что делать: запомнить `BASE`
и держать в голове, что он занижен ровно на долю документов длиннее окна энкодера.

<details><summary>Сколько мы потеряли на обрезке — оценка, которую стоило сделать</summary>

В разборе выше сказано, что база занижена долей документов длиннее окна энкодера, и что это
стоило бы измерить. Прикинем, насколько.

**Считаем.** Окно энкодера — 256 токенов, множитель токенов на слово около 1,35. Значит,
энкодер видит примерно первые 190 слов документа. Медиана длины у нас около 270 слов,
а верхний квартиль — за 450. То есть **больше половины документов обрезаются**, и у верхнего
квартиля отбрасывается больше половины текста.

**Что это значит для базы.** Вектор такого документа описывает его начало, а не его целиком.
Если вынутое предложение лежало во второй половине — а по нашей статистике медиана позиции
приходится примерно на 54 % документа, — то релевантный контекст в вектор не попал вовсе.
Примерно для половины запросов база отвечает не на тот вопрос, который мы ей задали.

**Как это меняет выводы.** База занижена, значит преимущество чанкования по полноте документа
**завышено**: часть выигрыша объясняется не тем, что чанки лучше, а тем, что база плохо
построена. Честное сравнение потребовало бы базы, где документ представлен несколькими
векторами по 190 слов, — то есть чанкованием с размером окна энкодера. Что, собственно,
и есть конфигурация 128 или 256 из нашей таблицы.

**Вывод, который из этого следует.** «Чанкование против отсутствия чанкования» — некорректная
постановка, и мы попали в неё сами. Корректная: «какой размер чанка», потому что **нулевого
варианта не существует**. Наша «база» — это чанк размером в окно энкодера с выбрасыванием
хвоста, то есть заведомо худшая из возможных конфигураций.

**Что стоило сделать.** Взять в качестве базы конфигурацию «один чанк на документ, но размером
в окно энкодера, и хвост не выбрасывается, а образует второй чанк». Это ровно наша строка 256
без перекрытия, и сравнивать надо было с ней. Разница выводов: преимущество мелких чанков
осталось бы, а «превосходство над базой» исчезло бы почти целиком.
</details>

<details><summary>Окно энкодера: почему 256 токенов — это фундаментальное ограничение, а не настройка</summary>

Мы обрезали документы по две тысячи символов, потому что энкодер всё равно не видит дальше
своего окна. Стоит понимать, откуда это окно берётся и почему его нельзя просто увеличить.

**Откуда 256 или 512.** Механизм внимания квадратичен по длине последовательности: удвоение
окна учетверяет вычисления и память. Модели вроде `all-MiniLM-L6-v2` обучены с окном 256 токенов
ради скорости — они предназначены для предложений и коротких абзацев, что прямо написано
в их названии.

**Что происходит с текстом длиннее.** Токенизатор обрезает **молча**. `encode` не выбросит
ни ошибки, ни предупреждения: он вернёт вектор, посчитанный по первым 256 токенам, и этот
вектор будет выглядеть совершенно нормально. Документ на тысячу слов кодируется по первой
пятой части.

**Как проверить у себя.** `model.max_seq_length` — одна строка. Сравни с медианной длиной
документов в токенах, и если вторая больше первой, ты уже теряешь текст.

**Модели с длинным окном.** Существуют: `jina-embeddings-v2` держит 8192 токена, `nomic-embed`
тоже. Они решают проблему обрезки и не решают более глубокую: **усреднение**. Вектор документа
в восемь тысяч токенов — это усреднение всего, что там есть, и он одинаково далёк от любого
конкретного запроса. Длинное окно позволяет **не терять** текст, но не позволяет **найти
фрагмент**.

**Отсюда чанкование как необходимость, а не оптимизация.** Даже с бесконечным окном пришлось
бы резать — иначе гранулярность поиска равна документу, и модель получит на вход двадцать
страниц ради одного абзаца. Это и есть содержание части 1: чанкование меняет **единицу
поиска**, и это его главный эффект, а не экономия.
</details>

⚠️ Ловушка A · **«Без чанкования» не существует.** Любой энкодер имеет конечное окно, и текст
длиннее него всё равно будет обрезан или разбит. Вопрос не в том, чанковать или нет, а в том,
делаешь ли ты это осознанно или это делает за тебя молчаливая обрезка в `encode`. Второе
происходит без единого предупреждения.

### Шаг 1.4 · две разные полноты

Теперь главное различение занятия. После чанкования вопрос «нашли ли мы нужное» распадается
на два **разных** вопроса с разными ответами:

* **полнота чанка** — попал ли в топ-k именно тот чанк, где лежал ответ. Это то, что увидит
  языковая модель.
* **полнота документа** — попал ли в топ-k хоть один чанк нужного документа. Это то, что
  показывают в статьях как «полнота системы».

In [ ]:
def build_chunks(size_w, overlap_w):
    chunks, owner = [], []
    gold = [set() for _ in range(N_QUERIES)]
    step = max(1, size_w - overlap_w)
    for di, d in enumerate(DOCS):
        w = d.split()
        for a in range(0, max(1, len(w)), step):
            piece = w[a:a + size_w]
            if not piece:
                break
            if di < N_QUERIES and a <= GOLD_W[di] < a + len(piece):
                gold[di].add(len(chunks))          # ВСЕ покрывающие чанки, не только первый
            chunks.append(" ".join(piece))
            owner.append(di)
            if a + size_w >= len(w):
                break
    return chunks, owner, gold

RESULTS, ORDERS = {}, {}
print(f"{'размер':>7} {'перекр':>7} {'чанков':>8} {'gold':>5} "
      f"{'R-чанка@5':>10} {'R-док@5':>9} {'R-док@5 базы':>13}")
for size_w, ov in CONFIGS:
    ch, ow, gold = build_chunks(size_w, ov)
    emb = st.encode(ch, batch_size=256, normalize_embeddings=True,
                    show_progress_bar=False).astype("float32")
    order = np.argsort(-(QEMB @ emb.T), axis=1)
    ORDERS[(size_w, ov)] = (order, gold, ow)

    def r_chunk(k):
        return float(np.mean([1.0 if gold[i] & set(order[i, :k].tolist()) else 0.0
                              for i in range(N_QUERIES)]))

    def r_doc(k):
        return float(np.mean([1.0 if i in {ow[c] for c in order[i, :k]} else 0.0
                              for i in range(N_QUERIES)]))

    RESULTS[(size_w, ov)] = {"n": len(ch), "gold": statistics.mean(len(g) for g in gold),
                             "chunk": {k: r_chunk(k) for k in (1, 3, 5, 10)},
                             "doc": {k: r_doc(k) for k in (1, 3, 5, 10)}}
    r = RESULTS[(size_w, ov)]
    print(f"{size_w:>7} {ov:>7} {len(ch):>8} {r['gold']:>5.1f} "
          f"{r['chunk'][5]:>10.3f} {r['doc'][5]:>9.3f} {BASE[5]:>13.3f}")
RUN["results"] = {f"{a}_{b}": v for (a, b), v in RESULTS.items()}

**Что видно.** Две колонки полноты отвечают на один и тот же вопрос **противоположно**.
полнота чанка не превосходит базу без чанкования **ни при одной** конфигурации, а полнота документа
превосходит её в большинстве. Сравнивать надо именно эти две колонки друг с другом: одна говорит
«чанкование ухудшило поиск», вторая — «улучшило», и обе посчитаны верно. Механизм в знаменателе
задачи: попасть в **конкретный** чанк труднее, чем в документ, потому что чанков в разы больше
и они конкурируют между собой; зато короткий чанк **точнее совпадает** с запросом по смыслу,
чем разбавленный документ целиком, и потому вытаскивает свой документ туда, куда целый документ
не пролезал. Чего эта таблица НЕ показывает: какая из двух колонок «правильная» — это зависит
от того, что делает следующая ступень. Что делать: всегда объявлять, какая из двух полнот
ты меряешь. Фраза «полнота выросла на десять процентов» без этого уточнения не значит ничего.

<details><summary>Полный разбор двух полнот: третья величина, о которой мы умолчали</summary>

В части 1 мы противопоставили полноту чанка и полноту документа. Есть третья, и она
на практике важнее обеих.

**Полнота контекста.** Собери топ-`k` чанков в один текст и спроси: содержится ли в нём ответ?
Ответ может быть собран из **нескольких** чанков — половина в одном, половина в соседнем.
Тогда ни один чанк не «золотой», полнота чанка равна нулю, а ответить модель сможет.

**Почему мы её не измерили.** Для неё нужна разметка на уровне утверждений: разбить эталонный
ответ на факты и проверить каждый на вхождение в контекст. Это ровно `context recall` из RAGAS,
и для его подсчёта на нашем корпусе нужна была бы либо ручная разметка, либо судья. У нас
нет ни того, ни другого, и мы честно посчитали его только на игрушке из лекции.

**Что это меняет.** Наша полнота чанка — **нижняя** оценка того, что реально доступно модели.
Настоящая доступность выше, потому что часть ответов собирается из соседних фрагментов. Значит,
все выводы части 1 о вреде мелких чанков — пессимистичны: мелкие чанки чаще требуют сборки
из нескольких, и именно этот случай мы не засчитывали.

**Как измерить дёшево.** Приблизительно: считать «попаданием» не золотой чанк, а **любой чанк,
пересекающийся с позицией ответа хотя бы на треть**. Это грубее разметки утверждений, но ловит
случай «ответ на границе» и стоит одной строки.

**Общий урок.** Всякий раз, когда метрика бинарна («попал или нет»), стоит спросить, не бывает
ли частичного попадания и что оно означает. В поиске по документам не бывает, в поиске
по фрагментам — бывает постоянно.
</details>

⚠️ Ловушка B · **полнота чанка и полнота документа — разные метрики с одним именем.** В статьях
по RAG почти всегда меряют второй, потому что он выше и его легче поднять. Модели же
скармливают чанки, то есть работает первый. Разница у нас доходит до пятнадцати пунктов,
и это не мелочь.

⚠️ Ловушка A · **При перекрытии золотых чанков несколько.** Позицию ответа покрывают два
соседних окна, и засчитывать надо **любое** из них. Прототип этого занятия сначала считал
только первое покрывающее — и полнота при перекрытии выходила заниженной, из-за чего
перекрытие выглядело вредным сильнее, чем оно есть. Ошибка была в разметке, а не в методе,
и не падала.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
sizes = sorted({s for s, _ in CONFIGS})
for j, key in enumerate(("chunk", "doc")):
    for ov_kind, style in (("без перекрытия", "o-"), ("с перекрытием", "s--")):
        xs, ys = [], []
        for s in sizes:
            match = [(a, b) for (a, b) in CONFIGS
                     if a == s and ((b == 0) == (ov_kind == "без перекрытия"))]
            if match:
                xs.append(s)
                ys.append(RESULTS[match[0]][key][5])
        ax[j].plot(xs, ys, style, label=ov_kind)
    ax[j].axhline(BASE[5], color="#B4521F", ls=":", label="без чанкования")
    ax[j].set_xscale("log", base=2)
    ax[j].set_xlabel("размер чанка, слов")
    ax[j].set_title("Recall ЧАНКА@5" if key == "chunk" else "Recall ДОКУМЕНТА@5")
    ax[j].legend(fontsize=8)
plt.suptitle("Один эксперимент, два вопроса, противоположные ответы")
plt.tight_layout(); plt.show()

**Что видно.** Сравнивать надо не кривые внутри панели, а **положение каждой относительно
оранжевого пунктира**: слева все кривые под ним, справа — в основном над. Это тот же факт,
что в таблице, но здесь видна форма: полнота чанка растёт с размером почти монотонно, полнота
документа имеет **максимум в середине** и падает к обоим краям. Ожидаемая картина по механизму:
чем крупнее чанк, тем он ближе к документу целиком, и обе кривые сходятся к базовой линии
справа; слева же мелкий чанк слишком узок, чтобы уверенно представлять свой документ. Чего
график НЕ показывает: цены — мелкие чанки дают в разы больше векторов, а значит больше памяти
и дольше индексация. Что делать: не выбирать размер по этому графику. Обе кривые построены
при **фиксированном k**, а это неправильная постановка — что и разбирается в следующей части.

<details><summary>Перекрытие: почему оно помогает меньше, чем принято думать</summary>

Народная мудрость велит всегда ставить перекрытие процентов в двадцать. Наши числа этого
не подтверждают, и разобраться полезно.

**Зачем перекрытие придумано.** Чтобы ответ, попавший на границу двух чанков, оказался
целиком хотя бы в одном. Аргумент верный и работает — но эффект количественный, а не
качественный: доля ответов, попадающих ровно на границу, невелика.

**Чем перекрытие платит.** Число чанков растёт: при перекрытии в четверть их становится
примерно на треть больше. Каждый новый чанк — это новый конкурент в выдаче, а поиск —
игра с фиксированным числом мест. Прирост покрытия борется с приростом конкуренции,
и знак итога заранее неизвестен.

**Плюс дублирование в контексте.** Два перекрывающихся чанка, оба попавшие в топ-`k`,
приносят модели один и тот же текст дважды. Бюджет потрачен, новой информации нет.
Это лечится дедупликацией — той самой, что мы писали на неделе 9, — и её почти никогда
не делают.

**Когда перекрытие действительно нужно.** Когда чанки маленькие относительно длины ответа.
Если ответ занимает пятьдесят слов, а чанк — шестьдесят, вероятность разрыва велика,
и перекрытие обязательно. Если чанк — пятьсот слов, разрыв редок, и перекрытие только плодит
дубликаты.

**Правило вместо мудрости.** Перекрытие имеет смысл порядка **типичной длины ответа**,
а не «двадцать процентов от чего бы то ни было». Оцени длину ответа на своих данных
и отталкивайся от неё.
</details>

---

## Часть 2 · Бюджет контекста — 20 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 2.1 | Почему сравнение при фиксированном k нечестно? | считаем, сколько контекста съедает каждая конфигурация |
| 2.2 | Что если фиксировать бюджет? | пересчитываем полноту при равных затратах |

Пять чанков по 512 слов и пять чанков по 64 слова — это разный объём контекста в восемь раз.
Сравнивать их при одном `k` — то же самое, что сравнивать системы с разной глубиной выдачи
на неделе 9.

In [ ]:
BUD = json.load(open(f"{DATA_DIR}/l10-budget.json", encoding="utf-8"))
tpw = RUN["tokens_per_word"]

print(f"бюджет: {BUDGET_WORDS} слов ≈ {BUDGET_WORDS * tpw:.0f} токенов")
print(f"{'размер':>7} {'k при бюджете':>15} {'слов':>7} {'токенов':>9}")
budget_k = {}
for size_w, ov in CONFIGS:
    if ov:
        continue
    k = max(1, BUDGET_WORDS // size_w)
    budget_k[size_w] = k
    print(f"{size_w:>7} {k:>15} {k * size_w:>7} {k * size_w * tpw:>9.0f}")
print(f"\nформула лекции для числа чанков в окне: {BUD['formula']}")
RUN["budget_k"] = budget_k

**Что видно.** При одном и том же бюджете число чанков различается на порядок: шестнадцать
мелких против двух крупных. Сравнивать надо не размеры чанков, а **что получает модель
на входе**: в обоих случаях примерно тысяча слов, но в первом это шестнадцать разных мест
корпуса, во втором — два. Механизм арифметический и жёсткий: окно контекста фиксировано,
и всё, что ты тратишь на размер чанка, ты не тратишь на их число. Чего эта таблица
НЕ показывает: что шестнадцать мест лучше двух — у мелких чанков меньше связного контекста
вокруг ответа, и модель может не понять фрагмент, вырванный из середины абзаца. Что делать:
считать бюджет **до** выбора размера, а не после. Именно он превращает «какой размер лучше»
в вопрос с ответом.

<details><summary>Что делать, когда чанков нужно много: иерархия и сжатие контекста</summary>

Часть 2 упёрлась в жёсткий выбор: либо крупные чанки, либо много мелких. Есть два способа
обойти эту дилемму, и оба применяются в проде.

**Иерархическое суммирование (RAPTOR).** Чанки кластеризуются, каждый кластер суммируется
моделью, суммы кластеризуются снова — получается дерево. Поиск идёт по всем уровням: на верхних
лежат обзорные summary, на нижних — исходные фрагменты. Запрос «о чём этот корпус» найдёт
верхний уровень, запрос про конкретную деталь — нижний. `data/l10-raptor.json` описывает
конструкцию.

Плата: построение дерева требует прогнать модель по всему корпусу, то есть это самая дорогая
из известных схем индексации. Плюс summary — это пересказ, и в нём уже потеряна часть деталей.

**Сжатие контекста.** Найденные чанки перед подачей модели прогоняются через фильтр, который
выбрасывает предложения, не относящиеся к запросу. Из шестнадцати чанков по шестьдесят слов
остаётся восемь предложений, и в бюджет влезает вдвое больше **полезного**. Фильтр может быть
дешёвым — кросс-энкодер на уровне предложений, тот самый с недели 7.

**Родительские документы (parent document retrieval).** Ищем по мелким чанкам, а в контекст
подставляем **родительский** крупный фрагмент. Это ровно тот случай, о котором говорит решение
задания 1: поиск работает указателем, и мерить надо полноту документа, а не чанка. Схема
популярна и хорошо работает, потому что берёт лучшее от обеих сторон нашей дилеммы.

**Что выбрать.** Родительские документы — первое, что стоит попробовать: реализуется
за полчаса, не требует ни модели, ни переиндексации. Сжатие контекста — второе. RAPTOR —
когда корпус стабилен, а запросы бывают обзорными.
</details>

<details><summary>Lost in the middle: почему больше контекста не значит лучше ответ</summary>

Мы всё занятие исходили из того, что попадание нужного фрагмента в контекст — это хорошо.
Так и есть, но с оговоркой, которая меняет выбор `k`.

**Измеренный эффект.** Работа Liu et al. 2023 показала: модель использует информацию из начала
и конца контекста заметно лучше, чем из середины. Кривая «точность против позиции нужного
фрагмента» имеет форму буквы U. При длинном контексте фрагмент, оказавшийся в середине, может
быть проигнорирован, даже если он там есть.

**Что это значит для `k`.** Увеличивая число чанков, ты повышаешь вероятность, что нужный
попадёт в контекст, и одновременно повышаешь вероятность, что он окажется в середине
и будет проигнорирован. Существует оптимум, и он обычно меньше, чем «сколько влезет».

**Что это значит для порядка.** Раз позиция важна, порядок чанков в промпте — это тоже
ранжирование, и его надо выбирать осознанно. Практический приём: класть самый релевантный
чанк **последним**, ближе к вопросу, а не первым. Ещё приём: класть по убыванию релевантности
с обоих концов к середине, оставляя в середине наименее важное.

**Связь с частью 2.** Наша таблица бюджета молча предполагала, что все `k` чанков одинаково
полезны. Это не так, и потому «шестнадцать чанков по шестьдесят четыре слова» на практике
хуже, чем выглядит по полноте: половина из них окажется в мёртвой зоне.

**Чего мы не измерили и почему.** Для измерения этого эффекта нужна генерация и разметка
качества ответов, то есть ровно то, чего мы сегодня избегали. Эффект известен и измерен
другими; мы им пользуемся как цитируемым знанием и говорим это вслух.
</details>

In [ ]:
print(f"{'размер':>7} {'k':>4} {'R-чанка@k':>10} {'R-док@k':>9} {'при k=5: R-док':>15}")
budget_rows = {}
for size_w in sorted(budget_k):
    k = budget_k[size_w]
    order, gold, ow = ORDERS[(size_w, 0)]
    rc = float(np.mean([1.0 if gold[i] & set(order[i, :k].tolist()) else 0.0
                        for i in range(N_QUERIES)]))
    rd = float(np.mean([1.0 if i in {ow[c] for c in order[i, :k]} else 0.0
                        for i in range(N_QUERIES)]))
    budget_rows[size_w] = {"k": k, "chunk": rc, "doc": rd}
    print(f"{size_w:>7} {k:>4} {rc:>10.3f} {rd:>9.3f} {RESULTS[(size_w, 0)]['doc'][5]:>15.3f}")
base_k = max(1, BUDGET_WORDS // int(RUN['median_doc_words']))
print(f"\nбез чанкования при том же бюджете: k={base_k}, R-док@{base_k} = {recall_doc(base_k):.3f}")
RUN["budget_rows"] = {str(k): v for k, v in budget_rows.items()}

**Что видно.** При равном бюджете разброс между конфигурациями вырастает втрое. Сравнивать
надо четвёртую колонку (`R_док` при бюджетном k) с последней (`R_док` при k=5): обе считают
долю найденных документов, и только их можно класть рядом — у третьей, полноты по чанкам,
другой знаменатель. Дальше по строкам: при фиксированном `k` все размеры лежали в узкой полосе,
а при фиксированном бюджете мелкие уходят далеко вверх, а самые крупные — далеко вниз.
Победитель тоже меняется. Механизм в том, что при фиксированном `k` мы бесплатно раздавали
крупным чанкам вчетверо больше контекста, и они этим пользовались; как только контекст уравняли,
преимущество исчезло, а вместе с ним и половина их полноты. Чего эта
таблица НЕ показывает: качества ответа — шестнадцать обрывков по шестьдесят слов могут быть
хуже двух связных абзацев, даже если нужный фрагмент найден чаще. Что делать: помнить,
что полнота при фиксированном бюджете — **правильная** постановка для поиска, но она всё ещё
не отвечает на вопрос про генерацию. Это два разных вопроса, и второй мы сегодня не меряем.

⚠️ Ловушка E · **Сравнение при фиксированном `k` систематически льстит крупным чанкам.**
Это ровно та же ошибка, что на неделе 9, где каскад с выдачей в двадцать пять документов
сравнивался с системами по сто. Единица честного сравнения — потраченный ресурс, а не число
элементов.

⚠️ Ловушка C · **Полнота выросла — не значит, ответ станет лучше.** Мелкий чанк чаще попадает
в цель и несёт меньше контекста вокруг цели. Мы меряем первое и не меряем второе, и потому
не имеем права заключать про качество ответа. Это самое частое незаконное умозаключение
в работах по RAG.

---

## Часть 3 · Метрики RAG без судьи — 30 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 3.1 | Что меряет RAGAS? | считаем все четыре метрики вручную и сверяем с лекцией |
| 3.2 | Какие из них не требуют модели? | разделяем метрики по тому, что для них нужно |

RAGAS — набор из четырёх метрик. Две из них про поиск, две про ответ, и это разделение
важнее самих формул.

In [ ]:
RG = json.load(open(f"{DATA_DIR}/l11-ragas.json", encoding="utf-8"))

# 1. Faithfulness: доля утверждений ответа, подтверждённых контекстом
supported = sum(1 for c in RG["answerClaims"] if c["supported"])
faithfulness = supported / len(RG["answerClaims"])

# 2. Answer relevance: средний косинус обратных вопросов с исходным
answer_relevance = statistics.mean(RG["reverseQuestionCos"])

# 3. Context precision: средняя precision@k по позициям релевантных контекстов
rel_positions = [p for p in RG["precisionAtK"] if p["relevant"]]
context_precision = statistics.mean(p["precisionAtK"] for p in rel_positions)

# 4. Context recall: доля утверждений эталона, покрытых контекстом
in_ctx = sum(1 for c in RG["groundTruthClaims"] if c["inContext"])
context_recall = in_ctx / len(RG["groundTruthClaims"])

for name, got, want in (("faithfulness", faithfulness, RG["faithfulness"]),
                        ("answer relevance", answer_relevance, RG["answerRelevance"]),
                        ("context precision", context_precision, RG["contextPrecision"]),
                        ("context recall", context_recall, RG["contextRecall"])):
    assert abs(got - want) < 1e-3, f"{name} разошлась с лекцией: {got:.4f} против {want}"

print(f"вопрос: {RG['question']}")
print(f"\n{'метрика':>20} {'значение':>10} {'что нужно, чтобы посчитать':>44}")
print(f"{'faithfulness':>20} {faithfulness:>10.4f} {'разбор ответа на утверждения -- НУЖЕН судья':>44}")
print(f"{'answer relevance':>20} {answer_relevance:>10.4f} {'генерация обратных вопросов -- НУЖЕН судья':>44}")
print(f"{'context precision':>20} {context_precision:>10.4f} {'только разметка релевантности':>44}")
print(f"{'context recall':>20} {context_recall:>10.4f} {'разметка + разбор эталона на утверждения':>44}")
print("\nсверка с data/l11-ragas.json: 4 значения совпали")
RUN["ragas"] = {"faithfulness": faithfulness, "answer_relevance": answer_relevance,
                "context_precision": context_precision, "context_recall": context_recall}

**Что видно.** Четыре метрики посчитаны, и правая колонка важнее левой. Сравнивать надо
не значения между собой, а **цену каждой метрики**: две верхние требуют языковой модели
в роли судьи — она разбивает ответ на утверждения и генерирует обратные вопросы, — а
context precision считается по одной лишь разметке релевантности, то есть по тем же `qrels`,
что на неделе 4. Механизм разделения прямой: метрики про **поиск** нуждаются только в разметке,
метрики про **ответ** нуждаются в модели. Чего эта таблица НЕ показывает: устойчивости
судейских метрик — судья это модель со своими предпочтениями, и часть 4 показывает, как легко
их сдвинуть. Что делать: строить оценку RAG снизу вверх. Сначала context precision и context recall,
которые дёшевы и воспроизводимы; судейские метрики — потом, и с оговорками.

<details><summary>Агентный RAG: когда одного прохода поиска не хватает</summary>

Всё занятие мы предполагали один проход: запрос — поиск — контекст — ответ. Есть класс запросов,
где этого мало по построению.

**Многошаговые вопросы.** «В каком городе родился режиссёр фильма, получившего Оскар
в 1994 году?» Одним поиском не решается: сначала надо найти фильм, потом режиссёра, потом город.
Никакое чанкование не поможет — нужного документа, содержащего ответ целиком, попросту
не существует.

**ReAct.** Модель чередует рассуждение и действие: думает, формулирует поисковый запрос,
получает результат, думает снова. Цикл повторяется, пока не наберётся достаточно фактов.
Реализуется просто, стоит нескольких вызовов модели на запрос и легко зацикливается, если
не поставить лимит шагов.

**Self-RAG.** Модель сама решает, нужен ли поиск, и сама оценивает полученные фрагменты
на релевантность специальными токенами. Экономит вызовы там, где поиск не нужен вовсе,
и добавляет обучение — модель приходится дообучать под эти токены.

**CRAG.** Лёгкий оценщик проверяет качество найденного и, если оно низкое, запускает запасной
путь — например, веб-поиск или переформулировку. `data/l10-selfrag.json` описывает пороги.
Практичная схема: не требует дообучения основной модели, только маленький классификатор.

**Что общего у всех трёх.** Они превращают поиск из одного вызова в **цикл**, и вместе с этим
приносят все проблемы циклов: недетерминированность, переменную задержку, стоимость, зависящую
от запроса. Оценивать их метриками из части 3 нельзя напрямую — контекст формируется
динамически, и «context recall» надо считать по объединению всех шагов.

**Когда это оправдано.** Когда доля многошаговых запросов заметна. Измеряется просто: возьми
сто запросов из логов и посчитай, для скольких ответ содержится в **одном** документе. Если
для девяноста — агентность не нужна и добавит только задержку.
</details>

<details><summary>Что не так с метриками, у которых судья — модель</summary>

Две из четырёх метрик RAGAS требуют языковой модели. Это не техническая деталь, а источник
проблем, которые стоит понимать до внедрения.

**Проблема первая: невоспроизводимость.** Судья — модель с версией, температурой и промптом.
Смени любое — числа поедут. Работа, опубликовавшая свои значения faithfulness в прошлом году,
измеряла другим судьёй, и повторить её невозможно даже с её же кодом.

**Проблема вторая: судья и генератор родственны.** Если ответы генерирует GPT, а судит тоже
GPT, возникает самопредпочтение: модель выше оценивает тексты своего стиля. Эффект измерен
и составляет проценты. Лечение — судья из другого семейства, но тогда возникает вопрос,
чьи предпочтения правильные.

**Проблема третья: разбиение на утверждения — само по себе задача.** Faithfulness требует
разбить ответ на атомарные факты. Модель делает это по-разному от запуска к запуску, и число
утверждений в знаменателе гуляет. Дисперсия метрики от этого растёт сильнее, чем от чего-либо
ещё.

**Что делать практически.** Первое: фиксировать судью по версии, промпту и температуре нулём,
и записывать это рядом с числами — ровно как мы записываем конфигурацию прогона. Второе:
не сравнивать свои числа с чужими, только свои со своими. Третье и главное: строить оценку
на **дешёвых** метриках там, где можно. Context precision обходится одной разметкой релевантности; context recall требует ещё и эталонного ответа, разобранного на утверждения, — это видно в таблице выше. Обе воспроизводимы без судьи и отвечают на самый важный вопрос — попало ли нужное в контекст.

**Практический порядок.** Сначала измеряй поиск метриками недели 4 на своей разметке. Только
если поиск хорош, а ответы плохи, зови судью. В обратном порядке ты будешь оптимизировать
генерацию поверх сломанного поиска и потратишь месяцы.
</details>

⚠️ Ловушка C · **Метрика context recall у RAGAS считается по утверждениям эталонного ответа,
а не по документам.** Это не та полнота, которую мы мерили в частях 1 и 2. Утверждение может
быть покрыто контекстом, собранным из трёх разных чанков, ни один из которых — не
«золотой документ». Три разных величины с одним словом в названии — на этом занятии уже
третья пара, и путать их стоит дорого.

### Шаг 3.2 · Потолок контекста

Та же мысль, что на неделях 7, 9 и 10, в четвёртый раз: **последняя ступень ограничена
первой**. Здесь она принимает вид «context recall ограничивает всё, что может сказать модель».

In [ ]:
gt_total = len(RG["groundTruthClaims"])
gt_missing = [c["text"] for c in RG["groundTruthClaims"] if not c["inContext"]]
print(f"утверждений в эталонном ответе: {gt_total}")
print(f"покрыто контекстом: {gt_total - len(gt_missing)} · НЕ покрыто: {len(gt_missing)}")
for t in gt_missing:
    print(f"   не найдено в контексте: {t!r}")
print(f"\nпотолок полноты ответа = context recall = {context_recall:.4f}")
print("модель физически не может сказать то, чего нет в контексте --")
print("остаётся либо промолчать, либо выдумать")
RUN["context_ceiling"] = context_recall

**Что видно.** Одно утверждение эталона в контекст не попало, и это ставит потолок всему,
что модель может ответить правильно. Сравнивать надо не метрики ответа между собой,
а **любую из них с этим потолком**: faithfulness может быть единицей при context recall
в две трети — ответ будет безупречно обоснован и при этом неполон. Механизм тот же,
что и во всех предыдущих каскадах, и формулировка та же: последняя ступень ограничена первой.
Чего этот замер НЕ показывает: что делает модель с непокрытым утверждением — промолчит
или придумает; первое честно, второе называется галлюцинацией, и различает их как раз
faithfulness. Что делать: при плохих ответах смотреть на context recall **первым**. Если он
низок, менять модель бессмысленно.

<details><summary>Галлюцинации: что это на самом деле и почему их не чинят промптом</summary>

Слово «галлюцинация» употребляют так широко, что оно перестало что-либо различать. Разложим
на три разных явления, потому что лечатся они по-разному.

**Первое: ответ не подкреплён контекстом.** Модель написала факт, которого в поданных чанках
нет. Это ровно то, что меряет `faithfulness`, и это **дефект генерации**: контекст был,
модель им не воспользовалась. Лечится промптом («отвечай только по контексту»), понижением
температуры и, лучше всего, требованием цитировать — заставить модель приводить фрагмент,
на который она опирается.

**Второе: контекст неполон.** Нужного факта в чанках не было, и модель заполнила пробел
правдоподобным. Это **дефект поиска**, и никаким промптом он не чинится: у модели нет
информации, а требование «не выдумывай» превращает ответ в «не знаю». Меряется `context recall`,
и мы весь семинар занимались именно этим.

**Третье: контекст противоречив.** Два чанка говорят разное — например, документ и его
устаревшая версия. Модель выберет один, обычно тот, что ближе к концу промпта. Это дефект
**данных**, и он самый коварный: и поиск, и генерация формально сработали правильно.

**Почему различение важно практически.** При жалобе «модель врёт» первое, что надо сделать, —
посмотреть на поданный контекст. Если факт там был — работай с промптом и моделью. Если
не было — работай с поиском, и никакие настройки генерации не помогут. Если было два
противоречащих — работай с корпусом.

**Дешёвая диагностика.** Логируй контекст вместе с ответом. Двадцать жалоб, разобранных
по этой схеме, разложатся по трём корзинам, и станет видно, где на самом деле проблема.
Почти всегда оказывается, что во второй, — то есть в поиске, а не в модели.
</details>

<details><summary>Потолок в четвёртый раз: одна мысль, четыре занятия</summary>

Стоит собрать вместе, потому что это самая переиспользуемая мысль курса.

**Неделя 7, каскад.** Потолок переранжирования равен `Recall@d` первой ступени. Кросс-энкодер
не может поставить первым документ, которого нет среди кандидатов. Мы реализовали две трети
потолка.

**Неделя 9, слияние.** Потолок объединения равен доле запросов, где ответ нашла хотя бы одна
из систем. Никакое слияние не превысит его. Мы его не взяли из-за обрезки по глубине.

**Неделя 10, приближённый поиск.** Потолок задан тем, что вернул индекс. Доуточнение
не создаёт документов, которых нет в отобранных кандидатах.

**Неделя 12, RAG.** Потолок полноты ответа равен `context recall`. Модель не скажет того,
чего нет в контексте.

**Одна форма.** В каждом случае система — конвейер, и каждая следующая ступень работает
с тем, что дала предыдущая. Ни одна ступень не может добавить информации, только упорядочить
или отфильтровать имеющуюся. Отсюда правило: **улучшать надо самое узкое место, а найти его
можно, посчитав потолок каждой ступени.**

**Почему это стоит делать первым.** Потолок считается за одну ячейку, не требует ни обучения,
ни подбора, и сразу говорит, где предел. Мы четыре раза начинали с него и четыре раза
экономили этим время. Если у тебя останется от курса одна привычка, пусть это будет она.

**Обратная сторона.** Потолок — не обещание. Расстояние между потолком и достигнутым говорит
о качестве **текущей** ступени, и оно может быть велико. Оба числа нужны вместе: потолок
без достигнутого не говорит, стоит ли улучшать отбор; достигнутое без потолка не говорит,
есть ли куда расти.
</details>

---

## Часть 4 · Судья и Гудхарт — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 4.1 | Как устроен судья на рубрике? | считаем оценки по критериям на данных лекции |
| 4.2 | Что происходит при перекосе рубрики? | меняем веса и получаем другого победителя |

Когда метрика требует судью, у судьи появляется рубрика — набор критериев с весами. Рубрика
кажется технической деталью. Она определяет победителя.

In [ ]:
J = json.load(open(f"{DATA_DIR}/l11-judge.json", encoding="utf-8"))
G = J["goodhart"]
crit = J["rubric"]["criteria"]

good, gamed = G["goodScores"], G["gamedScores"]
honest_good = statistics.mean(good)
honest_gamed = statistics.mean(gamed)

# перекос: критерий "полнота" (прокси длины) получает двойной вес
w = [1.0] * len(crit)
w[crit.index("completeness")] = 2.0
def weighted(scores):
    return sum(s * x for s, x in zip(scores, w)) / sum(w)
biased_good, biased_gamed = weighted(good), weighted(gamed)

assert abs(honest_good - G["honest"]["good"]) < 1e-3, "честное среднее разошлось с лекцией"
assert abs(biased_gamed - G["lengthBiased"]["gamed"]) < 1e-3, "смещённое среднее разошлось"

print(f"критерии рубрики: {crit}")
print(f"{'ответ':>10} " + " ".join(f"{c:>13}" for c in crit) +
      f" {'честно':>9} {'с перекосом':>13}")
print(f"{'A (хороший)':>10} " + " ".join(f"{s:>13}" for s in good) +
      f" {honest_good:>9.4f} {biased_good:>13.4f}")
print(f"{'C (нагнутый)':>10} " + " ".join(f"{s:>13}" for s in gamed) +
      f" {honest_gamed:>9.4f} {biased_gamed:>13.4f}")
print(f"\nпобедитель при честных весах:   {G['honest']['winner']}")
print(f"победитель при двойном весе полноты: {G['lengthBiased']['winner']}")
print("сверка с data/l11-judge.json: оба средних совпали")
RUN["judge"] = {"honest_winner": G["honest"]["winner"], "biased_winner": G["lengthBiased"]["winner"]}

**Что видно.** Одни и те же оценки по одним и тем же критериям дают **разных победителей**
в зависимости от весов. Сравнивать надо не строки между собой, а **две правые колонки одной
строки**: ни одна оценка не изменилась, изменилась только рубрика. Механизм прозрачен: критерий
«полнота» коррелирует с длиной ответа, и удвоив его вес, мы фактически стали премировать
многословность. Ответ C выигрывает не тем, что лучше, а тем, что длиннее. Чего этот пример
НЕ показывает: что перекос всегда преднамеренный — обычно он случаен, и рубрику пишут,
не думая о том, какой прокси в неё зашит. Что делать: смотреть на рубрику как на функцию
потерь. Всё, что ты в неё положил, будет оптимизировано, включая то, что положил случайно.

<details><summary>Шесть типов ловушек этого занятия — и почему тут снова доминирует C</summary>

Соберём сегодняшние ловушки вместе.

**A · данных.** «Без чанкования» не существует: энкодер обрезает молча. Плюс золотых чанков
при перекрытии несколько, и засчитывать надо любой — на этом споткнулся прототип занятия.

**B · метрики.** полнота чанка и полнота документа — разные величины с одним именем, и разница
у нас доходит до пятнадцати пунктов.

**C · интерпретации.** Три штуки: полнота выросла ≠ ответ станет лучше; context recall у RAGAS
считается по утверждениям, а не по документам, и это третья величина с тем же словом; судья
имеет собственные предпочтения помимо рубрики.

**D · инструмента.** Токены против слов: эвристика «четыре символа на токен» для русского
систематически врёт.

**E · замера.** Сравнение при фиксированном `k` льстит крупным чанкам — та же ошибка, что
на неделе 9 с разной глубиной выдачи.

**F · переноса.** Опубликованные числа RAGAS несравнимы между работами, потому что зависят
от судьи.

**Почему снова доминирует C.** Потому что сегодня три раза встретилось **одно слово в трёх
значениях**: полнота чанка, полнота документа, context recall. Каждое посчитано верно, каждое
отвечает на свой вопрос, и все три в литературе называются одинаково. Это не небрежность
авторов — это следствие того, что RAG собран из компонентов, у каждого из которых своя
традиция именования.

Отсюда правило занятия: **в RAG первым делом выясняй, что именно считает метрика — не как
она называется.** Цена ошибки здесь выше, чем в чистом поиске, потому что ступеней больше
и путать есть что.
</details>

<details><summary>Ограничения этого семинара, которые надо назвать вслух</summary>

Полный список того, где мы срезали угол, и в какую сторону это смещает выводы.

**Мы не меряли генерацию вовсе.** Ни одного сгенерированного ответа за занятие. Все выводы —
про поисковую половину, и ни один из них не переносится на качество ответов автоматически.
Это самое большое ограничение, и оно сознательное: генерация на бесплатном Colab
невоспроизводима, а с внешним API — платна и недетерминирована.

**Метрики RAGAS посчитаны на игрушке, а не на нашем корпусе.** Для них нужна разметка
утверждений, которой у нас нет. Мы проверили, что понимаем формулы, и не более того.

**Полнота чанка занижена.** Мы засчитывали только покрывающий чанк и не учитывали случай,
когда ответ собирается из двух соседних. Смещение в сторону «мелкие чанки хуже», и величина
смещения неизвестна.

**Псевдозапросы.** Те же, что на неделях 7–10, со всеми смещениями: лексически щедры,
длиннее реальных, ровно один релевантный документ.

**Одна модель эмбеддингов с окном 256 токенов.** Все выводы про размер чанка привязаны
к этому окну. На модели с окном 8192 оптимум был бы другим, и, возможно, чанкование выглядело
бы иначе.

**Эффект «lost in the middle» не измерен**, только процитирован. Он бьёт по выводам части 2:
шестнадцать мелких чанков в контексте, вероятно, хуже, чем показывает наша полнота.

<summary>Как сделать правильно, если есть бюджет</summary>
Взять локальную модель уровня `flan-t5-base`, сгенерировать ответы на каждой конфигурации
чанкования, разметить сто ответов вручную по трём критериям и посмотреть, коррелирует ли
полнота чанка с качеством ответа. Это день работы и превращает всё занятие из «мы измерили
поиск» в «мы измерили то, ради чего поиск существует».
</details>

⚠️ Ловушка C · **Судья-модель имеет собственные предпочтения помимо рубрики.** Известные
и измеренные: длина (длиннее — выше оценка), позиция в парном сравнении (первый вариант
выигрывает чаще), собственный стиль (модель выше оценивает тексты, похожие на её собственные).
Первое лечится нормировкой по длине, второе — предъявлением обеих перестановок и усреднением,
третье не лечится вовсе, если судья и генератор — одна модель.

⚠️ Ловушка F · **Опубликованные числа RAGAS не сравнимы между работами.** Метрика зависит
от судьи, а судья — от версии модели и от промпта. Две работы с одинаковой на вид faithfulness могли
измерять разное. Внутри одной работы сравнение осмысленно, между работами — нет.

---

## Задания — 15 мин

**Про самопроверку честно:** пройденная самопроверка не гарантирует, что задание сделано
осмысленно, но проваленная гарантирует, что где-то ошибка.

### Задание 1 · Конфигурация под бюджет

**Что сделать.** Найди `best_doc` — размер чанка (без перекрытия), максимизирующий **Recall
документа** при фиксированном бюджете, и `best_chunk` — размер, максимизирующий **полнота чанка**
при том же бюджете. **Честно сравни** их и посчитай, во сколько раз различается число векторов
в индексе.

**Что нужно получить.** `best_doc`, `best_chunk` (`int`), `index_ratio` (`float`) — отношение
числа чанков у `best_doc` к числу чанков у `best_chunk`.

**Подсказка.** Всё лежит в `budget_rows[size]` (ключи `doc`, `chunk`) и `RESULTS[(size, 0)]["n"]`.

**Прочитай до запуска.** Все исходы содержательны:
* размеры различаются — две метрики требуют разных конфигураций, и выбирать надо, зная,
  что делает следующая ступень;
* совпали — на нашем корпусе обе метрики согласны, и решение упрощается;
* `index_ratio` заметно больше единицы — конфигурация, лучшая по полноте документа, дороже
  по памяти, и это часть цены.

**Формулировка вывода.** Не «размер N лучший», а: **какой вопрос надо задать про следующую
ступень**, чтобы выбор между этими двумя размерами стал определённым.

In [ ]:
# --- твой код: ЗАДАНИЕ 1 ---
best_doc = ...
best_chunk = ...
index_ratio = ...
# --- конец ---

sizes_no_ov = sorted(budget_rows)
assert best_doc in sizes_no_ov and best_chunk in sizes_no_ov, "оба размера -- из сетки бюджета"
assert budget_rows[best_doc]["doc"] == max(budget_rows[s]["doc"] for s in sizes_no_ov), \
    "best_doc не максимизирует Recall ДОКУМЕНТА при бюджете"
assert budget_rows[best_chunk]["chunk"] == max(budget_rows[s]["chunk"] for s in sizes_no_ov), \
    "best_chunk не максимизирует Recall ЧАНКА при бюджете"
assert index_ratio > 0, "отношение размеров индекса положительно"
print(f"по полноте ДОКУМЕНТА: размер {best_doc}, k={budget_rows[best_doc]['k']}, "
      f"R-док {budget_rows[best_doc]['doc']:.3f}")
print(f"по полноте ЧАНКА:     размер {best_chunk}, k={budget_rows[best_chunk]['k']}, "
      f"R-чанка {budget_rows[best_chunk]['chunk']:.3f}")
print(f"индекс различается в {index_ratio:.2f} раза "
      f"({RESULTS[(best_doc, 0)]['n']} против {RESULTS[(best_chunk, 0)]['n']} векторов)")
RUN["task1"] = {"best_doc": best_doc, "best_chunk": best_chunk, "index_ratio": index_ratio}

### Задание 2 · Своя рубрика и свой Гудхарт

**Тезис.** *Любой критерий рубрики, коррелирующий с наблюдаемым признаком ответа, превращает
судью в измеритель этого признака.* Проверим на третьем критерии.

**Что сделать.** Возьми оценки из `G["goodScores"]` и `G["gamedScores"]` и найди `flip_weight` —
наименьший вес критерия `relevance` (при весах остальных, равных единице), при котором
победитель меняется относительно честного среднего. Перебирай вес с шагом 0,1 от 1,0 до 5,0.

**Что нужно получить.** `flip_weight` (`float`) либо `None`, если перекос по `relevance`
победителя не меняет.

**Подсказка.** Функция `weighted` уже написана; ей нужен список весов в порядке `crit`.

**Прочитай до запуска.** Все исходы содержательны:
* нашёлся вес — рубрика уязвима и по этому критерию тоже;
* `None` — по `relevance` перекос не работает, потому что хороший ответ выигрывает по нему
  с запасом; это **тоже результат** и говорит, какой критерий надёжен;
* вес около единицы — победитель меняется почти сразу, то есть честное преимущество
  было мизерным.

**Формулировка вывода.** Не «рубрики ненадёжны», а: **какое свойство оценок** делает рубрику
устойчивой к перекосу весов.

In [ ]:
# --- твой код: ЗАДАНИЕ 2 ---
flip_weight = ...
# --- конец ---

honest_winner = "A" if honest_good > honest_gamed else "C"
assert flip_weight is None or 1.0 <= flip_weight <= 5.0, "вес перебирается от 1.0 до 5.0"
if flip_weight is not None:
    ww = [1.0] * len(crit)
    ww[crit.index("relevance")] = flip_weight
    wg = sum(s * x for s, x in zip(good, ww)) / sum(ww)
    wc = sum(s * x for s, x in zip(gamed, ww)) / sum(ww)
    assert ("A" if wg > wc else "C") != honest_winner, \
        "при найденном весе победитель НЕ поменялся -- это не точка перелома"
    print(f"победитель меняется при весе relevance = {flip_weight:.1f} "
          f"(было {honest_winner}, стало {'A' if wg > wc else 'C'})")
else:
    print("перекос по relevance победителя не меняет во всём диапазоне 1.0--5.0")
    print(f"честный победитель {honest_winner} держится по этому критерию с запасом")
RUN["task2"] = {"flip_weight": flip_weight}

### Задание 3 · Словами: две полноты

**Что сделать.** В части 1 полнота чанка оказалась **ниже** базы без чанкования, а полнота документа — **выше**. Ответь **словами** на два вопроса:

1. Объясни механизм: почему одно и то же чанкование одновременно ухудшает одну полноту
   и улучшает другую. Свяжи ответ с числом чанков и с тем, насколько чанк «сфокусирован».
2. Твоя система отдаёт найденные чанки языковой модели. какая из двух полнот надо мерить
   и почему? Приведи случай, когда правильный ответ — второй.

**Прочитай до запуска.** `assert` проверяет объём, замену заглушки и упоминание числа чанков.
Ответ без него не может объяснить механизм.

**Формулировка вывода.** Не «мерить надо полнота чанка», а: от какого свойства следующей ступени
зависит выбор.

In [ ]:
# --- твой код: ЗАДАНИЕ 3 ---
ANSWER = """
Впиши ответ сюда: минимум 80 слов, оба пункта, с упоминанием числа чанков.
"""
# --- конец ---

assert len(ANSWER.split()) >= 80, "ответ короче 80 слов -- два пункта так не уместить"
assert "Впиши ответ" not in ANSWER, "заглушка не заменена"
assert "чанк" in ANSWER.lower(), "механизм невозможно объяснить, не сказав про число чанков"
print(f"ответ принят: {len(ANSWER.split())} слов")

---

## Итог занятия — 5 мин

* Показали, что «без чанкования» не существует: энкодер обрезает длинный документ молча,
  и вопрос лишь в том, кто принимает решение — ты или `encode`.
* Обнаружили, что **полнота чанка и полнота документа дают противоположные ответы** на вопрос
  «помогло ли чанкование». Обе метрики верны; путать их дорого.
* Сравнили конфигурации при **фиксированном бюджете контекста**, а не при фиксированном `k`,
  и картина перевернулась ещё раз.
* Посчитали все четыре метрики RAGAS вручную и разделили их по цене: точность контекста
  обходится разметкой, полнота требует ещё и разобранного эталона, обе генеративные — судьи.
* Увидели потолок контекста в четвёртый раз за курс: модель не может сказать того, чего нет
  в контексте.
* Поменяли вес одного критерия рубрики и получили другого победителя при неизменных оценках.

**Ограничение нашего замера, которое надо назвать вслух.** Мы **не меряли генерацию вовсе**.
Всё занятие — про поисковую половину RAG, и все выводы про полноту не переносятся на качество
ответа автоматически. Псевдозапросы те же, со всеми смещениями недель 7–10. Метрики RAGAS
посчитаны на одном игрушечном примере из лекции, а не на нашем корпусе: для них нужна разметка
утверждений, которой у нас нет.

**Что мы будем и чего не будем замерять дальше.** На неделе 14 всё собранное за курс —
индекс, каскад, слияние, чанкование — станет твоим проектом, и там впервые придётся выбрать
**одну** конфигурацию и защитить этот выбор числами.

<details><summary>Почему мы не мерили генерацию — и что бы это стоило</summary>

Самое большое ограничение занятия названо в итоге. Здесь — почему решение именно такое
и во что обошлось бы обратное.

**Что нужно, чтобы честно померить генерацию.** Модель, детерминированная настройка
(температура ноль и фиксированный сид, что не гарантирует воспроизводимости на всех бэкендах),
разметка правильных ответов на своём корпусе и метрика их сравнения. Каждый из четырёх пунктов
дорог, а последний ещё и спорен: сравнивать свободный текст с эталоном нечем, кроме судьи,
у которого свои смещения.

**Что дала бы локальная модель.** `flan-t5-base` весит 250 МБ и работает на CPU. Ответы были бы
короткие и посредственные, но **сравнимые между конфигурациями чанкования** — а нам нужно
именно сравнение, а не абсолютное качество. Это реалистичный путь, и он стоил бы часа счёта.

**Почему мы всё-таки не пошли.** Из-за разметки. Даже с моделью нужны эталонные ответы,
чтобы было с чем сравнивать, а у нас их нет: псевдозапросы дают правильный **документ**,
а не правильный **ответ**. Разметить полторы сотни ответов вручную — это несколько часов,
и они не помещаются в двухчасовое занятие.

**Что можно было сделать вместо.** Одну вещь дёшево: подать модели контексты из двух
конфигураций чанкования и попросить **выбрать**, какой контекст полезнее для ответа на вопрос.
Парное сравнение не требует эталона и требует судью — но судья здесь решает задачу проще,
чем оценка свободного текста, и потому надёжнее.

**Итог честно.** Мы измерили поисковую половину и **не измерили** ту половину, ради которой
поиск существует. Это ограничение, а не выбор темы: если бы у нас были эталонные ответы,
занятие выглядело бы иначе и было бы полнее. Проект недели 14 — подходящее место, чтобы
эту дыру закрыть на своём домене, где разметку можно собрать под конкретную задачу.
</details>

In [ ]:
RUN["finished"] = True
(ARTIFACTS / "rag.json").write_text(json.dumps({
    "chunk_size": best_doc, "overlap": 0, "k_at_budget": budget_rows[best_doc]["k"],
    "budget_words": BUDGET_WORDS, "tokens_per_word": RUN["tokens_per_word"],
    "recall_doc_at_budget": budget_rows[best_doc]["doc"],
    "recall_chunk_at_budget": budget_rows[best_doc]["chunk"],
    "base_no_chunking": BASE,
    "all_configs": RUN["results"],
    "run": RUN,
}, ensure_ascii=False), encoding="utf-8")
(ARTIFACTS / "run-rag.json").write_text(
    json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"чанкование -> {ARTIFACTS / 'rag.json'}")
print(f"замеры     -> {ARTIFACTS / 'run-rag.json'} ({len(RUN)} ключей)")
print(f"выбрано: чанк {best_doc} слов, k={budget_rows[best_doc]['k']} при бюджете "
      f"{BUDGET_WORDS} слов ≈ {BUDGET_WORDS * RUN['tokens_per_word']:.0f} токенов")
print("на неделе 14 это первая ступень твоего проекта")

**Что видно.** В артефакт легла конфигурация вместе с **бюджетом, при котором она выбрана**,
и с множителем токенов на слово. Сравнивать надо не размер чанка с чем-либо, а **конфигурацию
с условиями выбора**: тот же размер при другом бюджете окажется не лучшим, а множитель токенов
зависит от токенизатора и потому от модели. Механизм тот же, что с рабочей точкой недели 10:
конфигурация действительна для пары «корпус плюс модель» и невоспроизводима без неё.
Чего этот вывод НЕ показывает: что выбранная конфигурация даст лучшие **ответы** — мы их
не мерили и говорим это прямо. Что делать: на неделе 14 брать эту конфигурацию как отправную,
а не как окончательную, и перепроверять на своих данных.

<details><summary>Что защищать на неделе 14 — и чем плохая защита отличается от хорошей</summary>

Проект недели 14 — это не «сделать поиск», а показать, что каждое решение принято осознанно.
Разница видна сразу, и вот по каким признакам.

**Плохая защита.** «Мы взяли `all-MiniLM-L6-v2`, чанк 512 с перекрытием 20 %, HNSW, топ-5,
получили nDCG@10 = 0,64.» Ни одно из пяти решений не обосновано, число не с чем сравнить,
и неизвестно, лучше ли это, чем BM25 из недели 3.

**Хорошая защита.** «База — BM25, nDCG@10 = 0,41. Плотный поиск дал 0,44, интервал разницы
накрывает ноль — не отличимо, и мы оставили оба, потому что уникальный вклад лексики
на наших запросах составил 12 %. Чанк 128 выбран по полноте документа при фиксированном
бюджете в 1024 слова; 64 давал выше полноту, но вдвое больший индекс при том же качестве
ответа. Потолок объединения 0,88, реализовано 0,71 — узкое место в переранжировании,
и это следующий шаг.»

**Что различает эти два текста.** В первом — конфигурация и число. Во втором — **база,
интервал, потолок, цена и следующий шаг**. Ровно те пять вещей, которые мы считали каждую
неделю.

**Чего делать не надо.** Не надо строить самую сложную систему. Агентный RAG с четырьмя
ступенями, не измеренный ни на чём, хуже BM25, измеренного честно. Сложность защищается
числами, а не аргументом «так делают в проде».

**И одно про отрицательные результаты.** «Мы попробовали переранжирование, прирост оказался
внутри интервала, и мы его не внедрили» — это **сильная** часть защиты, а не слабая. Она
показывает, что ты умеешь останавливаться, а это в инженерии дороже умения добавлять.
</details>

---

## Решения

**Подглядеть — не поражение. Поражение — уйти с занятия, не поняв, где был затык.**

<details><summary>Задание 1 · конфигурация под бюджет</summary>

```python
sizes = sorted(budget_rows)
best_doc = max(sizes, key=lambda s: budget_rows[s]["doc"])
best_chunk = max(sizes, key=lambda s: budget_rows[s]["chunk"])
index_ratio = RESULTS[(best_doc, 0)]["n"] / RESULTS[(best_chunk, 0)]["n"]
```

Вопрос, который надо задать про следующую ступень: **видит ли она документ целиком или только
переданный чанк?** Если система после поиска подтягивает весь документ по идентификатору
(а так делают чаще, чем кажется), то важна полнота документа, и мелкие чанки выигрывают —
они работают как «указатели». Если модель получает ровно тот текст, что нашёл поиск,
важна полнота чанка, и слишком мелкие чанки вредны: нужный фрагмент найден, а контекста
вокруг него нет.

Отношение размеров индекса — часть цены этого выбора. Мелкие чанки дают в разы больше векторов,
а значит больше памяти, дольше индексацию и, как мы видели на неделе 10, другую рабочую точку
для ANN.
</details>

<details><summary>Задание 2 · перекос рубрики</summary>

```python
flip_weight = None
honest = "A" if statistics.mean(good) > statistics.mean(gamed) else "C"
for x in [round(1.0 + 0.1 * i, 1) for i in range(41)]:
    ww = [1.0] * len(crit)
    ww[crit.index("relevance")] = x
    wg = sum(s * v for s, v in zip(good, ww)) / sum(ww)
    wc = sum(s * v for s, v in zip(gamed, ww)) / sum(ww)
    if ("A" if wg > wc else "C") != honest:
        flip_weight = x
        break
```

Свойство, делающее рубрику устойчивой: **согласованность оценок между критериями**. Если
хороший ответ выигрывает по **всем** критериям, никакой перекос весов победителя не поменяет —
взвешенное среднее набора, доминирующего покомпонентно, всегда больше. Уязвимость возникает
ровно там, где оценки расходятся: один ответ лучше по одному критерию, другой по другому.

Отсюда практический приём: прежде чем спорить о весах, посмотри, есть ли вообще доминирование.
Если есть — веса не важны и обсуждать их пустая трата времени. Если нет — веса решают всё,
и их надо фиксировать до эксперимента, как метрику на неделе 4.
</details>

<details><summary>Задание 3 · две полноты</summary>

**Первый пункт.** Механизм состоит из двух встречных эффектов. Чанкование увеличивает **число
единиц** в индексе: вместо двух тысяч документов появляются тысячи чанков, и попасть
в конкретный из них труднее — конкурентов больше. Одновременно каждый чанк **сфокусирован**:
он про одну тему, а не про весь документ сразу, и потому его вектор ближе к вектору запроса,
чем усреднённый вектор всего документа. Первый эффект бьёт по полноте чанка, второй помогает
полноте документа, и потому одно и то же изменение двигает две метрики в разные стороны.

**Второй пункт.** Мерить надо ту полноту, которая соответствует тому, что получает следующая
ступень. Если модели передаётся текст найденных чанков — полноту чанка. Случай, когда правильный
ответ второй: система использует поиск как **указатель**, а в контекст подтягивает документ
целиком по идентификатору найденного чанка. Тогда неважно, какой именно чанк нашёлся, важно
лишь, что нашёлся документ, — и мерить надо полноту документа.

Общая форма ответа: выбор метрики определяется не устройством поиска, а устройством того,
что идёт после него. Ровно та же мысль, что на неделе 7 про первую ступень каскада.
</details>

---

## Литература

* **Lewis et al. (2020), «Retrieval-Augmented Generation»** — работа, давшая название. Полезна
  тем, что в ней поиск и генерация обучаются вместе, — подход, от которого практика отошла,
  и понимать почему полезно.
* **Es et al. (2023), «RAGAS: Automated Evaluation of Retrieval Augmented Generation»** —
  четыре метрики, которые мы считали руками. Читать вместе с критикой: метрики зависят
  от судьи, и авторы этого не скрывают.
* **Zheng et al. (2023), «Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena»** — измеренные
  систематические смещения судей: длина, позиция, самопредпочтение. Основание для ловушки
  в части 4.
* **Liu et al. (2023), «Lost in the Middle»** — как модель теряет информацию из середины
  длинного контекста. Прямо про то, почему «побольше чанков» не работает.
* **Günther et al. (2024), «Late Chunking»** — чанкование ПОСЛЕ кодирования, а не до;
  ровно про потерю контекста при разрезании. Это лекция L18 и семинар, которого у нас нет.
* **Лекции L15 и L16**, `data/l10-*.json` и `data/l11-*.json` — числа, с которыми мы сверялись.

**Дальше по курсу.** L18 показывает, как не терять контекст при разрезании; L20 — что делать,
когда одного прохода поиска не хватает. Неделя 14 собирает всё в один проект.

## Дамп прогона

Правило 10.5: занятие не считается прогнанным, пока его числа не лежат в файле рядом
с конфигурацией рантайма. Ячейка ниже собирает все численные результаты ноутбука —
от сида до финальных метрик — и кладёт их в `runs/hw-rag.json`. Это и есть
доказательство прогона: разбор сверяется с файлом, а не с памятью автора.

In [ ]:
# Итог прогона (правило 10.5): все числовые результаты + конфигурация рантайма +
# журнал печатей уезжают ОДНИМ архивом. Скачай его по ссылке ниже — и всё.
import json as _json, os as _os, sys as _sys, platform as _pl, shutil as _shutil, base64 as _b64

_runtime = {"python": _sys.version.split()[0], "platform": _pl.platform()}
_torch = _sys.modules.get("torch")   # НЕ импортируем сами: рамка 7.4 — сид и пин
if _torch is not None:                # обязателен только там, где ноутбук torch ИСПОЛЬЗУЕТ
    _runtime["torch"] = _torch.__version__
    _runtime["gpu"] = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else None
else:
    import subprocess as _sp
    try:
        _q = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=5)
        _runtime["gpu"] = (_q.stdout.strip().splitlines() or [None])[0] if _q.returncode == 0 else None
    except Exception:
        _runtime["gpu"] = None

def _plain(v):
    try:
        import numpy as _np
        if isinstance(v, _np.integer): return int(v)
        if isinstance(v, _np.floating): return float(v)
    except Exception:
        pass
    return v

def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)

def _tree(v, depth=0):
    """Числовое содержимое величины: само число, либо список/словарь чисел, в том числе
    ВЛОЖЕННЫЙ. Нечисловые ветки отбрасываются, пустое — тоже (None).

    Рекурсия здесь не украшение. Прежний сборщик брал словарь, только если ВСЕ его
    значения — числа, и поэтому целиком терял RUN: там лежит RUN["task1"] = {...},
    один вложенный словарь на весь ноутбук. А RUN — единственное место, где метрика
    записана в тех же единицах, в каких её печатает проза (RUN["bm25_ms"] = t_bm * 1000,
    тогда как в глобалах живут секунды). Из-за этого числа прозы не находились в дампе.
    """
    v = _plain(v)
    if _num(v):
        return v
    if depth >= 4:                     # защита от самоссылающихся структур
        return None
    if isinstance(v, dict) and 0 < len(v) <= 64:
        out = {}
        for _kk, _vv in v.items():
            got = _tree(_vv, depth + 1)
            if got is not None:
                out[str(_kk)] = got
        return out or None
    if isinstance(v, (list, tuple)) and 0 < len(v) <= 64:
        out = [_tree(_x, depth + 1) for _x in v]
        out = [_x for _x in out if _x is not None]
        return out or None
    return None

_metrics = {}
for _k, _v in sorted(globals().items()):
    if _k.startswith("_") or (len(_k) == 1 and _k.islower()):
        continue                       # служебные имена и счётчики циклов
    got = _tree(_v)
    if got is not None:
        _metrics[_k] = got

_payload = {"notebook": NB, "runtime": _runtime, "metrics": _metrics}
_json.dump(_payload, open(RUN_DIR / (NB + ".json"), "w", encoding="utf-8"),
           ensure_ascii=False, indent=1, sort_keys=True)
print(f"величин: {len(_metrics)} · рантайм: {_runtime['gpu'] or 'CPU'}")
_sys.stdout.flush(); _LOG.flush()      # журнал дописан до того, как попадёт в архив

_zip = _shutil.make_archive(str(RUN_DIR), "zip", str(RUN_DIR))
_kb = _os.path.getsize(_zip) / 1024
print(f"архив прогона: {_zip} ({_kb:.0f} КБ)")

# Ссылка на скачивание. Файл живёт в песочнице рантайма и сам до репозитория не доедет,
# а вывод ячейки — доедет: жми ссылку, архив упадёт в загрузки браузера.
_b = _b64.b64encode(open(_zip, "rb").read()).decode()
display(HTML(
    f'<a download="{_os.path.basename(_zip)}" href="data:application/zip;base64,{_b}" '
    f'style="display:inline-block;padding:10px 16px;margin:6px 0;background:#2b4a8b;'
    f'color:#fff;border-radius:4px;text-decoration:none;font-family:sans-serif">'
    f'&#10515; Скачать прогон — {_os.path.basename(_zip)} ({_kb:.0f} КБ)</a>'))
print("скачай архив по ссылке ↑, дальше локально: python3 scripts/import_runs.py")


**Что видно.** В дампе — конфигурация прогона и все скалярные результаты по именам
переменных. Сравнивать надо не тайминги — они свойство рантайма, и на T4, A100 и CPU
законно разные, — а метрики качества: при одном сиде они обязаны совпасть до знака.
Если твой прогон разошёлся с эталонным в качестве, а не во времени, — это находка,
неси её на занятие.